# 📚 From Zero to Researcher: Deep Neural Networks & LLMs
### A Complete Textbook Notebook — Math · Theory · Code

---

> **How to use this notebook:** Every section builds on the previous. Read the explanations, run every cell, and modify the code to build intuition. Treat this like an interactive textbook — come back to earlier sections as concepts click.

---

## 📋 Table of Contents

| # | Section | Topics |
|---|---------|--------|
| 1 | **Setup & Prerequisites** | Installing libraries |
| 2 | **Linear Algebra** | Vectors, matrices, operations, eigenvalues, SVD |
| 3 | **Calculus for ML** | Derivatives, chain rule, gradients, Jacobians |
| 4 | **Probability & Statistics** | Distributions, Bayes, MLE, entropy, KL divergence |
| 5 | **Optimization** | Gradient descent, momentum, Adam, loss landscapes |
| 6 | **Neural Networks from Scratch** | Perceptron, activation functions, forward pass, backprop |
| 7 | **Deep Networks** | Depth, initialization, batch norm, dropout, residuals |
| 8 | **Sequences & Recurrence** | RNNs, LSTMs, vanishing gradients |
| 9 | **Attention Mechanism** | Scaled dot-product attention, multi-head attention |
| 10 | **The Transformer** | Full architecture, positional encoding, layer norm, FFN |
| 11 | **Language Models & LLMs** | Tokenization, GPT, training objectives, scaling laws |
| 12 | **Research Foundations** | Mechanistic interpretability primer, superposition, SAEs |

---

**Prerequisites:** Basic Python (variables, loops, functions). Everything else is built from first principles here.

---
# Section 1: Setup & Prerequisites
---

In [ ]:
# Run this cell first — installs everything needed for the entire notebook
!pip install numpy matplotlib scipy sympy torch torchvision tqdm -q

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import scipy.linalg as la
import sympy as sp
from sympy import symbols, diff, sin, cos, exp, log, simplify, latex
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Plot styling
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#f8f9fa',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'font.size': 12,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print("✅ All libraries loaded. Let's begin.")
print(f"   PyTorch version: {torch.__version__}")
print(f"   NumPy version:   {np.__version__}")
print(f"   GPU available:   {torch.cuda.is_available()}")

---
# Section 2: Linear Algebra — The Language of Neural Networks
---

Neural networks are, at their core, sequences of **linear transformations** composed with **nonlinearities**. To understand them deeply, you must own linear algebra intuition, not just the mechanics.

## 2.1 Scalars, Vectors, Matrices, and Tensors

These are the four levels of the data hierarchy in ML:

| Object | Dimensions | Example | Notation |
|--------|-----------|---------|----------|
| **Scalar** | 0-D | a single temperature reading | $x \in \mathbb{R}$ |
| **Vector** | 1-D | word embedding, pixel row | $\mathbf{x} \in \mathbb{R}^n$ |
| **Matrix** | 2-D | weight layer, dataset | $\mathbf{W} \in \mathbb{R}^{m \times n}$ |
| **Tensor** | N-D | batch of images, attention scores | $\mathcal{T} \in \mathbb{R}^{d_1 \times d_2 \times ... \times d_k}$ |

**Why this matters for neural networks:** A layer with 512 inputs and 256 outputs is a matrix $\mathbf{W} \in \mathbb{R}^{256 \times 512}$. A batch of 32 sentences each tokenized to 128 tokens, each token embedded as a 768-dim vector, is a tensor of shape $(32, 128, 768)$.


In [ ]:
# =============================================================
# 2.1 — Scalars, Vectors, Matrices, Tensors in NumPy
# =============================================================

# --- Scalar ---
x = 3.14
print(f"Scalar: {x}, type: {type(x)}")

# --- Vector (1-D array) ---
v = np.array([1.0, 2.0, 3.0, 4.0])
print(f"\nVector: {v}")
print(f"  Shape: {v.shape}  | Dimensions: {v.ndim}  | Dtype: {v.dtype}")

# --- Matrix (2-D array) ---
W = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 9]], dtype=float)
print(f"\nMatrix W (shape {W.shape}):")
print(W)

# --- Tensor (3-D array) ---
# Imagine: batch_size=2, sequence_length=3, embedding_dim=4
T = np.random.randn(2, 3, 4)
print(f"\nTensor shape: {T.shape}  — think: (batch=2, seq_len=3, d_model=4)")
print(f"  Total elements: {T.size}")

## 2.2 Vector Operations — The Geometry of Data

### The Dot Product (Inner Product)

$$\mathbf{a} \cdot \mathbf{b} = \sum_{i=1}^{n} a_i b_i = |\mathbf{a}||\mathbf{b}|\cos(\theta)$$

This is one of the most important operations in all of deep learning. It measures **similarity** — specifically, how aligned two vectors are. A dot product of 0 means the vectors are orthogonal (perpendicular, no similarity). A large positive dot product means highly aligned.

**In attention mechanisms:** Query $\mathbf{q}$ and Key $\mathbf{k}$ vectors are dot-producted to measure "how relevant is this key to my query?" — you'll see this in Section 9.

### Norms — Measuring Vector Length

$$||\mathbf{v}||_2 = \sqrt{\sum_i v_i^2} \quad \text{(L2 norm, Euclidean length)}$$

$$||\mathbf{v}||_1 = \sum_i |v_i| \quad \text{(L1 norm, Manhattan distance)}$$

**In regularization:** L1 and L2 norms appear in loss functions to penalize large weights and prevent overfitting. L2 regularization (weight decay) is used in almost every modern LLM training run.

### Cosine Similarity

$$\text{cos}(\mathbf{a}, \mathbf{b}) = \frac{\mathbf{a} \cdot \mathbf{b}}{||\mathbf{a}|| \cdot ||\mathbf{b}||}$$

This normalizes the dot product to $[-1, 1]$, making it purely about direction, not magnitude. It's how you compare word embeddings — are "king" and "queen" similar in meaning?

In [ ]:
# =============================================================
# 2.2 — Vector Operations with Geometric Visualization
# =============================================================

a = np.array([3.0, 1.0])
b = np.array([1.0, 3.0])
c = np.array([3.0, 0.0])

# Dot products
print("=== Dot Products ===")
print(f"a · b = {np.dot(a, b):.2f}  (somewhat similar)")
print(f"a · c = {np.dot(a, c):.2f}  (more aligned with a)")

# Norms
print("\n=== Norms ===")
print(f"||a||₂ = {np.linalg.norm(a):.4f}  (L2/Euclidean)")
print(f"||a||₁ = {np.linalg.norm(a, 1):.4f}  (L1/Manhattan)")

# Cosine similarity
def cosine_similarity(x, y):
    return np.dot(x, y) / (np.linalg.norm(x) * np.linalg.norm(y))

print("\n=== Cosine Similarities ===")
print(f"cos(a, b) = {cosine_similarity(a, b):.4f}  (not very similar in direction)")
print(f"cos(a, c) = {cosine_similarity(a, c):.4f}  (more directionally aligned)")
print(f"cos(a, a) = {cosine_similarity(a, a):.4f}  (identical direction = 1.0)")

# Geometric visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Vectors and their geometric relationship
ax = axes[0]
origin = np.zeros(2)
vectors = {'a': (a, '#e74c3c'), 'b': (b, '#3498db'), 'c': (c, '#2ecc71')}
for name, (vec, color) in vectors.items():
    ax.quiver(*origin, *vec, angles='xy', scale_units='xy', scale=1,
              color=color, width=0.02, label=f'${name}$ = {vec}')
ax.set_xlim(-0.5, 4.5); ax.set_ylim(-0.5, 4.5)
ax.set_aspect('equal'); ax.legend(fontsize=10)
ax.set_title('Vectors in 2D Space', fontweight='bold')
ax.axhline(0, color='k', linewidth=0.5); ax.axvline(0, color='k', linewidth=0.5)

# Annotate angle between a and b
theta_ab = np.degrees(np.arccos(cosine_similarity(a, b)))
ax.text(2, 2.3, f'θ(a,b)={theta_ab:.1f}°', fontsize=9, color='purple')

# Plot 2: Cosine similarity demo — embedding space analogy
ax2 = axes[1]
# Simulate word embeddings in 2D
words = {'king': np.array([0.8, 0.6]),
         'queen': np.array([0.75, 0.65]),
         'man': np.array([0.7, 0.2]),
         'woman': np.array([0.65, 0.25]),
         'tree': np.array([-0.6, 0.5]),
         'forest': np.array([-0.7, 0.6])}

colors_w = ['#e74c3c','#e74c3c','#3498db','#3498db','#2ecc71','#2ecc71']
for i, (word, vec) in enumerate(words.items()):
    ax2.quiver(0, 0, vec[0], vec[1], angles='xy', scale_units='xy', scale=1,
               color=colors_w[i], width=0.015, alpha=0.8)
    ax2.text(vec[0]+0.02, vec[1]+0.02, word, fontsize=10, fontweight='bold',
             color=colors_w[i])

ax2.set_xlim(-1, 1.1); ax2.set_ylim(-0.1, 1.0)
ax2.set_aspect('equal')
ax2.set_title('Word Embeddings: Cosine Similarity = Semantic Closeness', fontweight='bold')
ax2.axhline(0, color='k', linewidth=0.5); ax2.axvline(0, color='k', linewidth=0.5)

# Print cosine sims for words
print("\n=== Word Embedding Cosine Similarities ===")
print(f"cos(king, queen)  = {cosine_similarity(words['king'], words['queen']):.4f}  ← Very similar!")
print(f"cos(man, woman)   = {cosine_similarity(words['man'], words['woman']):.4f}  ← Similar")
print(f"cos(king, tree)   = {cosine_similarity(words['king'], words['tree']):.4f}  ← Dissimilar")

plt.tight_layout()
plt.show()

## 2.3 Matrix Multiplication — The Core Operation of Neural Networks

Given $\mathbf{A} \in \mathbb{R}^{m \times k}$ and $\mathbf{B} \in \mathbb{R}^{k \times n}$, their product $\mathbf{C} = \mathbf{AB} \in \mathbb{R}^{m \times n}$ is:

$$C_{ij} = \sum_{l=1}^{k} A_{il} B_{lj}$$

**Three equivalent interpretations of matrix multiplication:**

1. **Dot product view:** $C_{ij}$ is the dot product of row $i$ of $\mathbf{A}$ with column $j$ of $\mathbf{B}$.
2. **Linear combination view:** Each column of $\mathbf{C}$ is a linear combination of columns of $\mathbf{A}$.
3. **Transformation view:** Matrix $\mathbf{A}$ transforms a vector, then $\mathbf{B}$... wait, it's $\mathbf{A}$ that transforms the columns of $\mathbf{B}$.

**Why this is the entire neural network:** A single linear layer does:

$$\mathbf{z} = \mathbf{W}\mathbf{x} + \mathbf{b}$$

Where $\mathbf{W}$ is the weight matrix, $\mathbf{x}$ is the input, $\mathbf{b}$ is the bias. That's it. The "intelligence" of neural networks comes from stacking many such operations with nonlinearities between them and learning the right $\mathbf{W}$ values.

In [ ]:
# =============================================================
# 2.3 — Matrix Multiplication: From First Principles to Neural Layer
# =============================================================

# --- Manual matmul to build intuition ---
def matmul_manual(A, B):
    """Matrix multiply from scratch — shows the inner loop."""
    m, k = A.shape
    k2, n = B.shape
    assert k == k2, "Inner dimensions must match!"
    C = np.zeros((m, n))
    for i in range(m):
        for j in range(n):
            for l in range(k):
                C[i, j] += A[i, l] * B[l, j]  # C_ij = sum_l A_il * B_lj
    return C

A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)  # shape (2, 3)
B = np.array([[7, 8],
              [9, 10],
              [11, 12]], dtype=float)  # shape (3, 2)

C_manual = matmul_manual(A, B)
C_numpy  = A @ B  # @ is Python's matrix multiply operator

print(f"A shape: {A.shape}   B shape: {B.shape}   C = A@B shape: {C_numpy.shape}")
print(f"\nC (manual) =\n{C_manual}")
print(f"C (numpy)  =\n{C_numpy}")
print(f"\nAre they equal? {np.allclose(C_manual, C_numpy)}")

# --- Simulating a single neural network linear layer ---
print("\n" + "="*50)
print("Simulating a Linear Layer: z = Wx + b")
print("="*50)

# Input: batch of 4 samples, each with 8 features
batch_size, input_dim, output_dim = 4, 8, 5

x = np.random.randn(batch_size, input_dim)    # (4, 8)  — input batch
W = np.random.randn(output_dim, input_dim) * 0.1  # (5, 8)  — weights
b = np.zeros(output_dim)                      # (5,)    — biases

# Forward pass of a linear layer
z = x @ W.T + b  # (4, 8) @ (8, 5) + (5,) = (4, 5)

print(f"Input x:       shape {x.shape}  — 4 samples, 8 features each")
print(f"Weight W:      shape {W.shape}  — maps 8 features → 5 outputs")
print(f"Bias b:        shape {b.shape}")
print(f"Output z=xWᵀ+b: shape {z.shape}  — 4 samples, 5 outputs each")
print(f"\nOutput z:\n{np.round(z, 4)}")
print("\n💡 This is literally ALL that happens in a linear layer!")

## 2.4 Eigenvalues and Eigenvectors — The "Skeleton" of a Matrix

For a square matrix $\mathbf{A}$, an **eigenvector** $\mathbf{v}$ and **eigenvalue** $\lambda$ satisfy:

$$\mathbf{A}\mathbf{v} = \lambda \mathbf{v}$$

This means: when you apply the transformation $\mathbf{A}$ to the vector $\mathbf{v}$, the result is just $\mathbf{v}$ **scaled** by $\lambda$ — not rotated, just stretched or flipped.

**Geometric intuition:** Most vectors change direction when multiplied by a matrix. Eigenvectors are the special directions that *don't change direction* — they only change in magnitude.

**Why do researchers care?**
- **PCA (Principal Component Analysis):** The principal components are eigenvectors of the covariance matrix. PCA is used constantly in ML for dimensionality reduction and analysis.
- **Gradient flow stability:** The eigenvalues of the weight matrix determine whether gradients explode or vanish during backprop (more on this in Section 7).
- **Attention heads as linear maps:** Analyzing what subspaces attention heads operate in uses eigendecomposition.
- **Mechanistic interpretability:** Researchers decompose weight matrices using SVD (below) to find interpretable structure.

## 2.5 Singular Value Decomposition (SVD) — Most Important for ML Research

**Any** matrix $\mathbf{A} \in \mathbb{R}^{m \times n}$ can be decomposed as:

$$\mathbf{A} = \mathbf{U} \mathbf{\Sigma} \mathbf{V}^T$$

Where:
- $\mathbf{U} \in \mathbb{R}^{m \times m}$ — left singular vectors (orthogonal matrix)
- $\mathbf{\Sigma} \in \mathbb{R}^{m \times n}$ — singular values on diagonal (non-negative, sorted descending)
- $\mathbf{V} \in \mathbb{R}^{n \times n}$ — right singular vectors (orthogonal matrix)

**Intuition:** SVD says every matrix is doing 3 things:
1. **Rotate** the input space ($\mathbf{V}^T$)
2. **Stretch** along the new axes ($\mathbf{\Sigma}$)
3. **Rotate** into the output space ($\mathbf{U}$)

**The singular values** $\sigma_1 \geq \sigma_2 \geq ... \geq 0$ tell you how much each "direction" of the matrix matters. Most weight matrices in neural networks have a few large singular values — the matrix has **low effective rank**.

**LoRA (Low-Rank Adaptation)** — the dominant fine-tuning technique for LLMs — is directly based on this: instead of updating $\mathbf{W}$ (huge matrix), you only update $\mathbf{A}\mathbf{B}$ where $\mathbf{A}$ and $\mathbf{B}$ are thin matrices. The hypothesis: the update $\Delta\mathbf{W}$ lives in a low-rank subspace.

In [ ]:
# =============================================================
# 2.4 & 2.5 — Eigendecomposition, SVD, and Low-Rank Approximation
# =============================================================

# --- Eigendecomposition ---
print("=== Eigendecomposition ===")
A = np.array([[4, 2],
              [1, 3]], dtype=float)
eigenvalues, eigenvectors = np.linalg.eig(A)
print(f"Matrix A:\n{A}")
print(f"\nEigenvalues:  {eigenvalues}")
print(f"Eigenvectors (columns):\n{eigenvectors}")

# Verify: A @ v = lambda * v
for i in range(len(eigenvalues)):
    v = eigenvectors[:, i]
    lam = eigenvalues[i]
    Av = A @ v
    lv = lam * v
    print(f"  Eigenvector {i}: A@v = {Av.round(4)}, λv = {lv.round(4)}, match: {np.allclose(Av, lv)}")

# --- SVD: The Critical Tool ---
print("\n=== Singular Value Decomposition (SVD) ===")
# Create a matrix (think: a weight matrix in a neural network)
np.random.seed(0)
# Create a RANK-2 matrix intentionally to demonstrate low-rank structure
u_true = np.random.randn(6, 2)
v_true = np.random.randn(2, 8)
W = u_true @ v_true  # This is a rank-2 matrix, shape (6, 8)
W_noisy = W + 0.3 * np.random.randn(6, 8)  # Add noise (like a real weight matrix)

U, S, Vt = np.linalg.svd(W_noisy, full_matrices=False)
print(f"Matrix W shape: {W_noisy.shape}")
print(f"Singular values: {S.round(3)}")
print("  → First 2 are MUCH larger — the matrix is approximately rank-2!")

# Low-rank approximation — the LoRA idea
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

def low_rank_approx(U, S, Vt, k):
    """Reconstruct matrix using only top-k singular values."""
    return U[:, :k] @ np.diag(S[:k]) @ Vt[:k, :]

axes[0].imshow(W_noisy, cmap='RdBu', aspect='auto')
axes[0].set_title(f'Original Matrix\n(rank={np.linalg.matrix_rank(W_noisy)})', fontweight='bold')
axes[0].set_xlabel('columns'); axes[0].set_ylabel('rows')

W_rank2 = low_rank_approx(U, S, Vt, k=2)
err2 = np.linalg.norm(W_noisy - W_rank2) / np.linalg.norm(W_noisy)
axes[1].imshow(W_rank2, cmap='RdBu', aspect='auto')
axes[1].set_title(f'Rank-2 Approximation\n(error={err2:.3f})', fontweight='bold')
axes[1].set_xlabel('columns')

# Singular value spectrum
axes[2].bar(range(len(S)), S, color=['#e74c3c' if i < 2 else '#95a5a6' for i in range(len(S))])
axes[2].set_title('Singular Value Spectrum', fontweight='bold')
axes[2].set_xlabel('Singular value index')
axes[2].set_ylabel('Magnitude')
red_patch = mpatches.Patch(color='#e74c3c', label='Used in rank-2 approx')
gray_patch = mpatches.Patch(color='#95a5a6', label='Discarded (mostly noise)')
axes[2].legend(handles=[red_patch, gray_patch])

plt.suptitle('SVD & Low-Rank Approximation — The Idea Behind LoRA', fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 LoRA insight: Instead of updating a big W directly, we update W + A@B")
print("   where A ∈ R^(d×r) and B ∈ R^(r×d), r << d")
print(f"   Example: W is 768×768 = {768*768:,} params")
print(f"   LoRA r=8: A is 768×8, B is 8×768 → {768*8 + 8*768:,} params (97% reduction!)")

---
# Section 3: Calculus for Machine Learning
---

Neural networks **learn** by adjusting their weights to minimize a loss function. The mechanism for doing this is calculus — specifically, the **gradient** tells us which direction to move the weights to reduce the loss.

## 3.1 Derivatives — The Instantaneous Rate of Change

The derivative of $f(x)$ at point $x$ measures how fast $f$ changes as $x$ changes:

$$f'(x) = \frac{df}{dx} = \lim_{h \to 0} \frac{f(x+h) - f(x)}{h}$$

**Geometric meaning:** The derivative at a point is the slope of the tangent line to $f$ at that point.

**Key derivatives to memorize:**

| Function $f(x)$ | Derivative $f'(x)$ | Why it matters in ML |
|---|---|---|
| $x^n$ | $nx^{n-1}$ | Polynomial layers |
| $e^x$ | $e^x$ | Exponential in softmax |
| $\ln(x)$ | $1/x$ | Log-likelihood loss |
| $\sigma(x) = 1/(1+e^{-x})$ | $\sigma(x)(1-\sigma(x))$ | Sigmoid activation |
| $\max(0, x)$ (ReLU) | $\mathbb{1}[x > 0]$ | Most used activation |

## 3.2 The Chain Rule — How Backpropagation Works

**This is the most important rule in all of deep learning.**

If $z = g(y)$ and $y = f(x)$, then:

$$\frac{dz}{dx} = \frac{dz}{dy} \cdot \frac{dy}{dx}$$

A neural network is a composition of many functions:

$$\text{output} = f_L(f_{L-1}(... f_2(f_1(\mathbf{x}))))$$

The chain rule lets us compute the gradient of the loss with respect to the very first layer's weights — even though they're far from the output. Backpropagation **is** the chain rule applied efficiently to a computation graph.

## 3.3 Partial Derivatives and Gradients

For a function of many variables $f(x_1, x_2, ..., x_n)$, the **partial derivative** $\frac{\partial f}{\partial x_i}$ measures how $f$ changes when only $x_i$ changes.

The **gradient** is the vector of all partial derivatives:

$$\nabla_\mathbf{x} f = \left[ \frac{\partial f}{\partial x_1}, \frac{\partial f}{\partial x_2}, ..., \frac{\partial f}{\partial x_n} \right]^T$$

**The gradient points in the direction of steepest ascent.** To minimize $f$, we go in the *negative* gradient direction: $\mathbf{x} \leftarrow \mathbf{x} - \alpha \nabla_\mathbf{x} f$. This is gradient descent.

In [ ]:
# =============================================================
# 3.1 — Derivatives: Symbolic and Numerical
# =============================================================

x = sp.Symbol('x')

functions = {
    'x³ - 2x² + x': x**3 - 2*x**2 + x,
    'sigmoid σ(x)': 1 / (1 + sp.exp(-x)),
    'log(x)': sp.log(x),
    'e^x': sp.exp(x),
}

print("=== Symbolic Derivatives ===")
for name, f in functions.items():
    df = sp.diff(f, x)
    df_simplified = sp.simplify(df)
    print(f"  d/dx [{name}] = {df_simplified}")

# Numerical derivative (finite differences) — how autograd works internally
print("\n=== Numerical Derivative (Finite Differences) ===")
def numerical_derivative(f, x0, h=1e-7):
    """Approximate derivative using the limit definition."""
    return (f(x0 + h) - f(x0)) / h

f = lambda x: x**3 - 2*x**2 + x
df_exact = lambda x: 3*x**2 - 4*x + 1  # Analytic derivative

x0 = 2.0
num_deriv = numerical_derivative(f, x0)
exact_deriv = df_exact(x0)
print(f"At x={x0}: numerical = {num_deriv:.6f}, exact = {exact_deriv:.6f}")
print(f"Error: {abs(num_deriv - exact_deriv):.2e}  (incredibly close!)")

In [ ]:
# =============================================================
# 3.2 — Chain Rule: Manual Backpropagation Through a Mini-Network
# =============================================================

print("Manual Backpropagation — Chain Rule in Action")
print("="*55)
print("Network: z = w*x + b  →  a = sigmoid(z)  →  L = (a - y)²")
print()

# Forward pass
w, b, x_val, y_true = 0.5, 0.1, 2.0, 1.0

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

z = w * x_val + b             # Linear: z = wx + b
a = sigmoid(z)                # Activation
L = (a - y_true)**2           # MSE Loss

print(f"FORWARD PASS:")
print(f"  z = w·x + b = {w}·{x_val} + {b} = {z}")
print(f"  a = σ(z) = σ({z:.3f}) = {a:.6f}")
print(f"  L = (a - y)² = ({a:.3f} - {y_true})² = {L:.6f}")

# Backward pass — chain rule
# dL/da = 2(a - y)
dL_da = 2 * (a - y_true)
# da/dz = sigmoid'(z) = σ(z)(1 - σ(z))
da_dz = a * (1 - a)
# dz/dw = x
dz_dw = x_val
# dz/db = 1
dz_db = 1.0

# Chain rule: dL/dw = dL/da * da/dz * dz/dw
dL_dw = dL_da * da_dz * dz_dw
dL_db = dL_da * da_dz * dz_db

print(f"\nBACKWARD PASS (chain rule):")
print(f"  ∂L/∂a  = 2(a-y) = {dL_da:.6f}")
print(f"  ∂a/∂z  = σ(z)·(1-σ(z)) = {da_dz:.6f}")
print(f"  ∂z/∂w  = x = {dz_dw}")
print(f"  ∂z/∂b  = 1")
print(f"")
print(f"  ∂L/∂w  = {dL_da:.4f} × {da_dz:.4f} × {dz_dw} = {dL_dw:.6f}")
print(f"  ∂L/∂b  = {dL_da:.4f} × {da_dz:.4f} × 1 = {dL_db:.6f}")

# Verify with numerical gradient
h = 1e-7
def loss(w_, b_):
    z_ = w_*x_val + b_
    a_ = sigmoid(z_)
    return (a_ - y_true)**2

dL_dw_num = (loss(w+h, b) - loss(w, b)) / h
dL_db_num = (loss(w, b+h) - loss(w, b)) / h
print(f"\nVERIFICATION (numerical gradient):")
print(f"  ∂L/∂w numerical: {dL_dw_num:.6f}  (matches: {np.isclose(dL_dw, dL_dw_num)})")
print(f"  ∂L/∂b numerical: {dL_db_num:.6f}  (matches: {np.isclose(dL_db, dL_db_num)})")

print("\n💡 This is EXACTLY what PyTorch's autograd does — but for millions of parameters!")

In [ ]:
# =============================================================
# 3.3 — Gradients: Visualizing the Loss Landscape
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# --- Loss landscape visualization ---
w_range = np.linspace(-3, 3, 200)
b_range = np.linspace(-3, 3, 200)
W_grid, B_grid = np.meshgrid(w_range, b_range)

# Compute loss over the grid
x_data, y_data = 2.0, 1.0
Z_grid = np.vectorize(loss)(W_grid, B_grid)

ax = axes[0]
contour = ax.contourf(W_grid, B_grid, Z_grid, levels=50, cmap='viridis')
plt.colorbar(contour, ax=ax, label='Loss')
ax.set_xlabel('w (weight)')
ax.set_ylabel('b (bias)')
ax.set_title('Loss Landscape L(w, b) — Darker = Lower Loss', fontweight='bold')

# Gradient descent path
w_curr, b_curr = -2.5, 2.5
lr = 0.5
path_w, path_b, path_l = [w_curr], [b_curr], [loss(w_curr, b_curr)]

for _ in range(30):
    # Compute gradients at current point
    z_ = w_curr * x_data + b_curr
    a_ = sigmoid(z_)
    dL_da_ = 2 * (a_ - y_data)
    da_dz_ = a_ * (1 - a_)
    g_w = dL_da_ * da_dz_ * x_data
    g_b = dL_da_ * da_dz_
    # Update
    w_curr -= lr * g_w
    b_curr -= lr * g_b
    path_w.append(w_curr)
    path_b.append(b_curr)
    path_l.append(loss(w_curr, b_curr))

ax.plot(path_w, path_b, 'r-o', markersize=4, label='Gradient descent path', linewidth=2)
ax.plot(path_w[0], path_b[0], 'g*', markersize=15, label='Start')
ax.plot(path_w[-1], path_b[-1], 'r*', markersize=15, label='End')
ax.legend()

# --- Loss over steps ---
axes[1].plot(path_l, 'b-', linewidth=2)
axes[1].set_xlabel('Gradient Descent Step')
axes[1].set_ylabel('Loss')
axes[1].set_title('Loss Decreasing During Training', fontweight='bold')
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

print(f"Final loss: {path_l[-1]:.2e}  (started at {path_l[0]:.4f})")

---
# Section 4: Probability & Statistics — The Foundation of Learning
---

## 4.1 Probability Distributions

A **probability distribution** describes how likely different outcomes are. In ML:
- **Model outputs** are distributions (e.g., probability over vocabulary tokens)
- **Data** comes from unknown distributions we're trying to model
- **Training** is about making the model's distribution match the data's distribution

**Continuous distributions (key ones):**

**Normal (Gaussian):** $\mathcal{N}(\mu, \sigma^2)$ — $p(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$
- Used for: weight initialization, noise modeling, many natural phenomena

**Uniform:** $\mathcal{U}(a, b)$ — equal probability everywhere in $[a, b]$

**Discrete distributions (key ones):**

**Categorical/Softmax:** probability over $K$ classes: $p(y=k) = \frac{e^{z_k}}{\sum_{j=1}^{K} e^{z_j}}$
- This is the **final layer** of every classification model and language model

## 4.2 Bayes' Theorem

$$P(A|B) = \frac{P(B|A) \cdot P(A)}{P(B)}$$

**In ML language:** $P(\theta | \text{data}) = \frac{P(\text{data} | \theta) \cdot P(\theta)}{P(\text{data})}$
- $P(\theta | \text{data})$ = posterior — what we believe about weights given data
- $P(\text{data} | \theta)$ = likelihood — how well the model explains the data
- $P(\theta)$ = prior — what we believed before seeing data

## 4.3 Maximum Likelihood Estimation (MLE)

**Core idea:** Find the parameters $\theta$ that maximize the probability of seeing your training data.

$$\hat{\theta}_{MLE} = \arg\max_\theta \prod_{i=1}^{N} p(x_i; \theta) = \arg\max_\theta \sum_{i=1}^{N} \log p(x_i; \theta)$$

(We take log to convert product to sum — easier to optimize, same answer)

**Training a language model is MLE:** We maximize $\sum_t \log p(\text{token}_t | \text{context}_t; \theta)$. That's it. The loss is negative log-likelihood, also called **cross-entropy loss**.

## 4.4 Information Theory — Entropy, Cross-Entropy, KL Divergence

**Entropy** measures uncertainty in a distribution:
$$H(P) = -\sum_x P(x) \log P(x)$$
- High entropy = uniform distribution (maximum uncertainty)
- Low entropy = peaked distribution (near certain)

**Cross-Entropy** measures how well distribution $Q$ approximates $P$:
$$H(P, Q) = -\sum_x P(x) \log Q(x)$$

This is your **training loss** in almost every neural network! $P$ is the true labels, $Q$ is model predictions.

**KL Divergence** measures how much $Q$ differs from $P$:
$$D_{KL}(P \| Q) = \sum_x P(x) \log \frac{P(x)}{Q(x)} = H(P, Q) - H(P)$$

KL divergence appears everywhere in modern ML: VAEs, RLHF (KL penalty from reference policy), knowledge distillation.

In [ ]:
# =============================================================
# 4.1-4.4 — Probability, MLE, Information Theory, and Softmax
# =============================================================

# --- Distributions ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

x_range = np.linspace(-5, 5, 1000)

# Normal distributions
ax = axes[0, 0]
for mu, sigma, color in [(0, 1, '#e74c3c'), (0, 2, '#3498db'), (2, 0.5, '#2ecc71')]:
    pdf = (1/(sigma*np.sqrt(2*np.pi))) * np.exp(-0.5*((x_range-mu)/sigma)**2)
    ax.plot(x_range, pdf, color=color, linewidth=2.5,
            label=f'μ={mu}, σ={sigma}')
ax.set_title('Gaussian Distributions N(μ, σ²)', fontweight='bold')
ax.legend(); ax.set_ylabel('p(x)')

# Softmax — the output of every classifier and LM
ax = axes[0, 1]
def softmax(z):
    e = np.exp(z - z.max())  # Numerically stable
    return e / e.sum()

logits_cool   = np.array([1.0, 2.0, 0.5, -0.5])   # low temperature
logits_hot    = np.array([1.0, 2.0, 0.5, -0.5])

# Temperature scaling: z/T   (T→0: argmax, T→∞: uniform)
tokens = ['cat', 'dog', 'bird', 'fish']
temps  = [0.3, 1.0, 3.0]
colors_t = ['#e74c3c', '#3498db', '#2ecc71']
x_pos = np.arange(len(tokens))
bar_width = 0.25

for i, (T, color) in enumerate(zip(temps, colors_t)):
    probs = softmax(logits_cool / T)
    ax.bar(x_pos + i*bar_width, probs, bar_width, label=f'Temperature T={T}', color=color, alpha=0.8)

ax.set_xticks(x_pos + bar_width)
ax.set_xticklabels(tokens)
ax.set_title('Softmax with Temperature Scaling', fontweight='bold')
ax.set_ylabel('Probability'); ax.legend()
ax.text(0.5, 0.85, 'T→0: argmax (greedy)\nT→∞: uniform (random)',
        transform=ax.transAxes, ha='center', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8))

# Entropy illustration
ax = axes[1, 0]
def entropy(p):
    p = np.clip(p, 1e-12, 1)
    return -np.sum(p * np.log2(p))

distributions = [
    ([0.25, 0.25, 0.25, 0.25], 'Uniform\n(max uncertainty)'),
    ([0.7, 0.1, 0.1, 0.1],    'Peaked\n(confident)'),
    ([0.99, 0.003, 0.004, 0.003], 'Very Peaked\n(near certain)'),
]
entropies = [entropy(d) for d, _ in distributions]
labels    = [name for _, name in distributions]
bars = ax.bar(labels, entropies, color=['#e74c3c', '#3498db', '#2ecc71'], alpha=0.85, edgecolor='black')
for bar, H in zip(bars, entropies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'H={H:.3f} bits', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Entropy — Measuring Uncertainty', fontweight='bold')
ax.set_ylabel('Entropy (bits)')

# Cross-Entropy Loss visualization
ax = axes[1, 1]
# True label: class 1 (one-hot [0, 1, 0, 0])
y_true_oh = np.array([0, 1, 0, 0])
# Different model predictions
pred_good = softmax(np.array([-1.0, 3.0, 0.5, -0.5]))
pred_bad  = softmax(np.array([2.0, -1.0, 1.0, 0.5]))
pred_ok   = softmax(np.array([0.5, 1.0, 0.3, 0.2]))

def cross_entropy(y_true, y_pred):
    return -np.sum(y_true * np.log(np.clip(y_pred, 1e-12, 1)))

preds  = {'Good prediction': pred_good, 'OK prediction': pred_ok, 'Bad prediction': pred_bad}
colors_ce = ['#2ecc71', '#f39c12', '#e74c3c']
ces    = [cross_entropy(y_true_oh, p) for p in preds.values()]

bars2 = ax.bar(list(preds.keys()), ces, color=colors_ce, alpha=0.85, edgecolor='black')
for bar, ce in zip(bars2, ces):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'CE={ce:.3f}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Cross-Entropy Loss — Your Training Objective', fontweight='bold')
ax.set_ylabel('Cross-Entropy Loss (lower = better)')

plt.tight_layout()
plt.show()

# KL Divergence calculation
print("=== KL Divergence ===")
def kl_divergence(P, Q):
    P, Q = np.clip(P, 1e-12, 1), np.clip(Q, 1e-12, 1)
    return np.sum(P * np.log(P / Q))

P = np.array([0.4, 0.3, 0.2, 0.1])  # True distribution
Q1 = np.array([0.35, 0.32, 0.22, 0.11])  # Close to P
Q2 = np.array([0.1, 0.1, 0.4, 0.4])      # Far from P

print(f"KL(P || Q1) = {kl_divergence(P, Q1):.4f}  ← Q1 is close to P")
print(f"KL(P || Q2) = {kl_divergence(P, Q2):.4f}  ← Q2 is far from P")
print(f"KL(P || P)  = {kl_divergence(P, P):.4f}   ← Always 0 for identical distributions")
print(f"\n💡 In RLHF: KL(π_RL || π_SFT) penalizes the RL model from diverging too much from SFT model")

---
# Section 5: Optimization — Teaching Networks to Learn
---

## 5.1 Gradient Descent

The fundamental learning algorithm. Given a loss $\mathcal{L}(\theta)$:

$$\theta_{t+1} = \theta_t - \alpha \nabla_\theta \mathcal{L}(\theta_t)$$

Where $\alpha$ is the **learning rate** — the most critical hyperparameter in training.

**Three flavors:**

| Variant | Update on... | Pros | Cons |
|---------|-------------|------|------|
| **BGD** (Batch) | All data | Accurate gradient | Too slow for large datasets |
| **SGD** (Stochastic) | 1 sample | Fast, noisy | Very high variance |
| **Mini-batch SGD** | Small batch | Best of both | Most used in practice |

## 5.2 Momentum — Remembering Past Gradients

Pure SGD oscillates a lot. **Momentum** accumulates a velocity vector:

$$v_{t+1} = \beta v_t + (1-\beta) \nabla_\theta \mathcal{L}$$
$$\theta_{t+1} = \theta_t - \alpha v_{t+1}$$

Typical $\beta = 0.9$ — means 90% of the velocity comes from history.

## 5.3 Adam — The Standard Optimizer

**Adam** (Adaptive Moment Estimation) adapts the learning rate for each parameter individually:

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t \quad \text{(first moment = mean of gradients)}$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2 \quad \text{(second moment = variance of gradients)}$$
$$\hat{m}_t = \frac{m_t}{1-\beta_1^t}, \quad \hat{v}_t = \frac{v_t}{1-\beta_2^t} \quad \text{(bias correction)}$$
$$\theta_{t+1} = \theta_t - \frac{\alpha}{\sqrt{\hat{v}_t} + \epsilon} \hat{m}_t$$

**Intuition:** Parameters with consistently large gradients get smaller effective learning rates (they're already moving fast). Parameters with small, uncertain gradients get relatively larger updates (they need more exploration).

**Default values:** $\beta_1=0.9, \beta_2=0.999, \epsilon=10^{-8}$. These work well across essentially all neural network training runs — Adam is remarkably robust.

**AdamW** adds weight decay *correctly* (decoupled from gradient adaptation) and is what's actually used to train modern LLMs.

In [ ]:
# =============================================================
# 5.1–5.3 — Comparing Optimizers on a Tricky Loss Landscape
# =============================================================

# Rosenbrock function — a classic difficult optimization landscape
# Minimum at (1, 1) with f(1,1) = 0
def rosenbrock(w, b, a=1, c=100):
    return (a - w)**2 + c * (b - w**2)**2

def rosenbrock_grad(w, b, a=1, c=100):
    dw = -2*(a - w) - 4*c*w*(b - w**2)
    db = 2*c*(b - w**2)
    return np.array([dw, db])

# --- Implement optimizers from scratch ---
class SGD:
    def __init__(self, lr=0.001):
        self.lr = lr
    def step(self, params, grads):
        return params - self.lr * grads

class SGDMomentum:
    def __init__(self, lr=0.001, beta=0.9):
        self.lr = lr; self.beta = beta
        self.v = None
    def step(self, params, grads):
        if self.v is None: self.v = np.zeros_like(grads)
        self.v = self.beta * self.v + (1 - self.beta) * grads
        return params - self.lr * self.v

class Adam:
    def __init__(self, lr=0.01, beta1=0.9, beta2=0.999, eps=1e-8):
        self.lr=lr; self.b1=beta1; self.b2=beta2; self.eps=eps
        self.m = None; self.v = None; self.t = 0
    def step(self, params, grads):
        if self.m is None:
            self.m = np.zeros_like(grads)
            self.v = np.zeros_like(grads)
        self.t += 1
        self.m = self.b1 * self.m + (1 - self.b1) * grads
        self.v = self.b2 * self.v + (1 - self.b2) * grads**2
        m_hat = self.m / (1 - self.b1**self.t)
        v_hat = self.v / (1 - self.b2**self.t)
        return params - self.lr * m_hat / (np.sqrt(v_hat) + self.eps)

# Run each optimizer
start = np.array([-1.5, 1.5])
n_steps = 2000

optimizers = [
    ('SGD (lr=0.001)', SGD(lr=0.001), '#e74c3c'),
    ('Momentum (lr=0.001)', SGDMomentum(lr=0.001), '#3498db'),
    ('Adam (lr=0.01)', Adam(lr=0.01), '#2ecc71'),
]

results = {}
for name, opt, color in optimizers:
    params = start.copy()
    path = [params.copy()]
    losses = [rosenbrock(*params)]
    for _ in range(n_steps):
        grads = rosenbrock_grad(*params)
        params = opt.step(params, grads)
        path.append(params.copy())
        losses.append(rosenbrock(*params))
    results[name] = (np.array(path), losses, color)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Loss landscape
w_r = np.linspace(-2, 2, 400)
b_r = np.linspace(-0.5, 3, 400)
W_r, B_r = np.meshgrid(w_r, b_r)
Z_r = rosenbrock(W_r, B_r)

ax = axes[0]
ax.contourf(W_r, B_r, np.log(Z_r + 1), levels=40, cmap='viridis', alpha=0.8)
ax.contour(W_r, B_r, np.log(Z_r + 1), levels=40, colors='white', alpha=0.2, linewidths=0.5)

for name, (path, losses, color) in results.items():
    ax.plot(path[:, 0], path[:, 1], color=color, linewidth=1.5, alpha=0.8, label=name)
    ax.plot(*path[0], 'w*', markersize=12)
    ax.plot(*path[-1], 'o', color=color, markersize=8)

ax.plot(1, 1, 'r*', markersize=15, label='Global minimum (1,1)')
ax.set_title('Optimizer Paths on Rosenbrock Function', fontweight='bold')
ax.legend(loc='lower right', fontsize=9); ax.set_xlabel('w'); ax.set_ylabel('b')

# Loss curves
ax2 = axes[1]
for name, (path, losses, color) in results.items():
    ax2.semilogy(losses, color=color, linewidth=2, label=name)
ax2.set_title('Loss Convergence Comparison', fontweight='bold')
ax2.set_xlabel('Step'); ax2.set_ylabel('Loss (log scale)')
ax2.legend()

plt.tight_layout()
plt.show()

print("\nFinal losses after 2000 steps:")
for name, (_, losses, _) in results.items():
    print(f"  {name:30s}: {losses[-1]:.6f}")

---
# Section 6: Neural Networks from Scratch
---

## 6.1 The Perceptron — The Simplest Neural Unit

A perceptron takes inputs $\mathbf{x}$, multiplies by weights $\mathbf{w}$, adds bias $b$, and passes through an activation function $\phi$:

$$\hat{y} = \phi(\mathbf{w}^T \mathbf{x} + b)$$

One perceptron can only learn **linearly separable** problems. Stack them into layers, and you get something much more powerful.

## 6.2 Activation Functions — Adding Nonlinearity

Without activation functions, stacking linear layers gives you... one linear layer. You need nonlinearities.

| Function | Formula | Key Property |
|----------|---------|-------------|
| **Sigmoid** | $1/(1+e^{-x})$ | Smooth, output ∈ (0,1). Suffers vanishing gradients. |
| **Tanh** | $(e^x - e^{-x})/(e^x + e^{-x})$ | Zero-centered, output ∈ (-1,1). Better than sigmoid. |
| **ReLU** | $\max(0, x)$ | Simple, doesn't saturate. The default for deep networks. |
| **GELU** | $x \cdot \Phi(x)$ | Smooth ReLU variant. Used in GPT, BERT. |
| **SiLU/Swish** | $x \cdot \sigma(x)$ | Self-gated. Used in Llama, PaLM. |

## 6.3 The Universal Approximation Theorem

> A neural network with at least one hidden layer of sufficient width (or sufficient depth) can approximate **any** continuous function to arbitrary precision.

This doesn't tell us *how* to find those weights (that's learning), or *how many* neurons we need, but it tells us the architecture is expressive enough in principle.

## 6.4 Forward Propagation — Making Predictions

For a 2-layer network:

$$\mathbf{h} = \phi(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) \quad \text{(hidden layer)}$$
$$\hat{\mathbf{y}} = \text{softmax}(\mathbf{W}_2 \mathbf{h} + \mathbf{b}_2) \quad \text{(output layer)}$$

## 6.5 Backpropagation — The Full Algorithm

Backpropagation is the chain rule applied systematically to compute $\nabla_\theta \mathcal{L}$:

**Step 1:** Forward pass — compute activations layer by layer, cache them.

**Step 2:** Backward pass — compute gradients layer by layer, from output to input:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_2} = \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{y}}} \cdot \mathbf{h}^T$$

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \left(\mathbf{W}_2^T \frac{\partial \mathcal{L}}{\partial \hat{\mathbf{y}}}\right) \odot \phi'(\mathbf{z}_1) \cdot \mathbf{x}^T$$

**Step 3:** Update weights: $\mathbf{W} \leftarrow \mathbf{W} - \alpha \nabla_\mathbf{W} \mathcal{L}$

In [ ]:
# =============================================================
# 6.2 — Activation Functions: Properties and Gradients
# =============================================================

x = np.linspace(-5, 5, 1000)

# Define activations and their derivatives
def gelu(x):
    from scipy.special import erf
    return 0.5 * x * (1 + erf(x / np.sqrt(2)))

activations = {
    'Sigmoid': {
        'f': lambda x: 1 / (1 + np.exp(-x)),
        'df': lambda x: (1/(1+np.exp(-x))) * (1 - 1/(1+np.exp(-x))),
        'color': '#e74c3c', 'note': 'Saturates at extremes → vanishing gradients'
    },
    'Tanh': {
        'f': np.tanh,
        'df': lambda x: 1 - np.tanh(x)**2,
        'color': '#3498db', 'note': 'Zero-centered; still saturates'
    },
    'ReLU': {
        'f': lambda x: np.maximum(0, x),
        'df': lambda x: (x > 0).astype(float),
        'color': '#2ecc71', 'note': 'No saturation for x>0; dead neuron problem'
    },
    'GELU': {
        'f': gelu,
        'df': lambda x: 0.5*(1 + np.tanh(np.sqrt(2/np.pi)*(x + 0.044715*x**3))) + \
                        x * 0.5*(1-np.tanh(np.sqrt(2/np.pi)*(x+0.044715*x**3))**2) * \
                        np.sqrt(2/np.pi)*(1 + 3*0.044715*x**2),
        'color': '#9b59b6', 'note': 'Smooth; used in GPT and BERT'
    },
    'SiLU/Swish': {
        'f': lambda x: x * (1/(1+np.exp(-x))),
        'df': lambda x: (1/(1+np.exp(-x))) + x*(1/(1+np.exp(-x)))*(1 - 1/(1+np.exp(-x))),
        'color': '#e67e22', 'note': 'Used in LLaMA, PaLM'
    },
}

fig, axes = plt.subplots(2, 5, figsize=(20, 8))

for i, (name, act) in enumerate(activations.items()):
    # Plot activation
    ax_f = axes[0, i]
    f_vals = act['f'](x)
    ax_f.plot(x, f_vals, color=act['color'], linewidth=2.5)
    ax_f.axhline(0, color='k', linewidth=0.5); ax_f.axvline(0, color='k', linewidth=0.5)
    ax_f.set_title(f'{name}', fontweight='bold', fontsize=13)
    ax_f.set_xlim(-5, 5)
    if i == 0: ax_f.set_ylabel('f(x)', fontsize=11)

    # Plot derivative
    ax_df = axes[1, i]
    df_vals = act['df'](x)
    ax_df.plot(x, df_vals, color=act['color'], linewidth=2.5, linestyle='--')
    ax_df.axhline(0, color='k', linewidth=0.5); ax_df.axvline(0, color='k', linewidth=0.5)
    ax_df.set_xlim(-5, 5)
    ax_df.set_xlabel('x')
    ax_df.text(0.5, -0.25, act['note'], transform=ax_df.transAxes,
               ha='center', fontsize=8, style='italic', wrap=True)
    if i == 0: ax_df.set_ylabel("f'(x)", fontsize=11)

plt.suptitle('Activation Functions (top) and Their Derivatives (bottom)', fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# 6.4-6.5 — A Complete Neural Network from Scratch (NumPy only!)
# Classification on a non-linearly-separable dataset
# =============================================================

from sklearn.datasets import make_moons

# Generate the famous "two moons" dataset — not linearly separable!
X_data, y_data = make_moons(n_samples=400, noise=0.2, random_state=42)
y_data_oh = np.eye(2)[y_data]  # One-hot: shape (400, 2)

# Normalize data
X_data = (X_data - X_data.mean(0)) / X_data.std(0)

class NeuralNetwork:
    """
    2-hidden-layer fully connected neural network.
    Architecture: input(2) → hidden(32) → hidden(32) → output(2)
    Activations: ReLU → ReLU → Softmax
    Loss: Cross-entropy
    Optimizer: Mini-batch SGD
    """
    
    def __init__(self, input_dim=2, hidden_dim=32, output_dim=2, lr=0.1):
        self.lr = lr
        # He initialization for ReLU activations
        # Keeps variance stable across layers
        scale = np.sqrt(2.0 / input_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim) * scale
        self.b1 = np.zeros(hidden_dim)
        
        scale = np.sqrt(2.0 / hidden_dim)
        self.W2 = np.random.randn(hidden_dim, hidden_dim) * scale
        self.b2 = np.zeros(hidden_dim)
        
        self.W3 = np.random.randn(hidden_dim, output_dim) * 0.01
        self.b3 = np.zeros(output_dim)
    
    def relu(self, z):
        return np.maximum(0, z)
    
    def relu_backward(self, z):
        return (z > 0).astype(float)  # 1 where z > 0, else 0
    
    def softmax(self, z):
        e = np.exp(z - z.max(axis=1, keepdims=True))  # Numerically stable
        return e / e.sum(axis=1, keepdims=True)
    
    def forward(self, X):
        """Forward pass: compute predictions and cache intermediate values."""
        self.X = X
        
        # Layer 1
        self.z1 = X @ self.W1 + self.b1      # (N, 32)
        self.h1 = self.relu(self.z1)          # (N, 32)
        
        # Layer 2
        self.z2 = self.h1 @ self.W2 + self.b2 # (N, 32)
        self.h2 = self.relu(self.z2)           # (N, 32)
        
        # Output layer
        self.z3 = self.h2 @ self.W3 + self.b3  # (N, 2)
        self.probs = self.softmax(self.z3)      # (N, 2)
        
        return self.probs
    
    def cross_entropy_loss(self, y_oh, probs):
        N = y_oh.shape[0]
        return -np.sum(y_oh * np.log(np.clip(probs, 1e-12, 1))) / N
    
    def backward(self, y_oh):
        """Backward pass: compute all gradients via chain rule."""
        N = y_oh.shape[0]
        
        # --- Output layer gradient ---
        # dL/dz3 = (probs - y_oh) / N  (combined softmax + cross-entropy gradient)
        dz3 = (self.probs - y_oh) / N                # (N, 2)
        dW3 = self.h2.T @ dz3                        # (32, 2)
        db3 = dz3.sum(axis=0)                        # (2,)
        
        # --- Hidden layer 2 gradient ---
        dh2 = dz3 @ self.W3.T                        # (N, 32)
        dz2 = dh2 * self.relu_backward(self.z2)      # (N, 32)
        dW2 = self.h1.T @ dz2                        # (32, 32)
        db2 = dz2.sum(axis=0)                        # (32,)
        
        # --- Hidden layer 1 gradient ---
        dh1 = dz2 @ self.W2.T                        # (N, 32)
        dz1 = dh1 * self.relu_backward(self.z1)      # (N, 32)
        dW1 = self.X.T @ dz1                         # (2, 32)
        db1 = dz1.sum(axis=0)                        # (32,)
        
        # --- SGD update ---
        self.W3 -= self.lr * dW3; self.b3 -= self.lr * db3
        self.W2 -= self.lr * dW2; self.b2 -= self.lr * db2
        self.W1 -= self.lr * dW1; self.b1 -= self.lr * db1
    
    def train_step(self, X, y_oh):
        probs = self.forward(X)
        loss  = self.cross_entropy_loss(y_oh, probs)
        self.backward(y_oh)
        return loss
    
    def predict(self, X):
        return self.forward(X).argmax(axis=1)

# --- Training loop ---
np.random.seed(42)
model = NeuralNetwork(lr=0.5)
n_epochs = 300
batch_size = 64
losses = []

for epoch in range(n_epochs):
    # Shuffle data
    idx = np.random.permutation(len(X_data))
    X_shuffled, y_shuffled = X_data[idx], y_data_oh[idx]
    
    epoch_loss = 0
    for i in range(0, len(X_data), batch_size):
        Xb = X_shuffled[i:i+batch_size]
        yb = y_shuffled[i:i+batch_size]
        epoch_loss += model.train_step(Xb, yb)
    
    losses.append(epoch_loss)

# Final accuracy
preds = model.predict(X_data)
acc = (preds == y_data).mean()

print(f"Training complete! Final accuracy: {acc*100:.1f}%")

# --- Visualization ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Data
axes[0].scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdBu', edgecolors='k', alpha=0.7, s=50)
axes[0].set_title('Dataset: Two Moons (non-linear!)', fontweight='bold')

# Decision boundary
h = 0.02
xx, yy = np.meshgrid(np.arange(-3, 3, h), np.arange(-3, 3, h))
grid_pts = np.c_[xx.ravel(), yy.ravel()]
Z = model.predict(grid_pts).reshape(xx.shape)

axes[1].contourf(xx, yy, Z, cmap='RdBu', alpha=0.5)
axes[1].scatter(X_data[:, 0], X_data[:, 1], c=y_data, cmap='RdBu', edgecolors='k', alpha=0.8, s=30)
axes[1].set_title(f'Learned Decision Boundary (acc={acc*100:.1f}%)', fontweight='bold')

# Training loss
axes[2].plot(losses, '#e74c3c', linewidth=2)
axes[2].set_title('Training Loss over Epochs', fontweight='bold')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Cross-Entropy Loss')

plt.tight_layout()
plt.show()

---
# Section 7: Deep Networks — Engineering for Depth
---

Making networks deeper is a double-edged sword. More layers = more expressivity, but also:
- **Vanishing gradients** — gradients shrink exponentially as they flow backward
- **Exploding gradients** — gradients grow exponentially
- **Slow training** — takes forever to converge

## 7.1 The Vanishing Gradient Problem

With sigmoid activations and deep networks, the chain rule multiplies many small numbers:

$$\frac{\partial \mathcal{L}}{\partial \mathbf{W}_1} = \frac{\partial \mathcal{L}}{\partial \mathbf{h}_L} \cdot \prod_{l=2}^{L} \mathbf{W}_l^T \cdot \phi'(\mathbf{z}_l)$$

Sigmoid's derivative maxes out at 0.25. With 10 layers: $0.25^{10} \approx 10^{-6}$ — essentially zero gradient in the early layers. They don't learn.

**Solutions:**
1. **ReLU** — derivative is 1 for positive inputs, doesn't saturate
2. **Residual connections** — add a shortcut that passes gradients directly
3. **Careful initialization** — control the initial scale of weights

## 7.2 Weight Initialization

**Xavier/Glorot** initialization (for tanh): $W \sim \mathcal{N}(0, \sqrt{\frac{2}{n_{in} + n_{out}}})$

**He/Kaiming** initialization (for ReLU): $W \sim \mathcal{N}(0, \sqrt{\frac{2}{n_{in}}})$

The idea: ensure the variance of activations and gradients stays roughly constant across layers.

## 7.3 Batch Normalization

Normalizes activations within each mini-batch:

$$\hat{x}_i = \frac{x_i - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}} \quad \text{then} \quad y_i = \gamma \hat{x}_i + \beta$$

Where $\gamma$ and $\beta$ are **learned** scale and shift parameters.

**Benefits:** Reduces internal covariate shift, allows higher learning rates, acts as slight regularization, makes training dramatically more stable.

**Note for Transformers:** Transformers use **Layer Normalization** instead — normalizes across the feature dimension (not batch). Critical for sequential data where batch statistics are unstable.

$$\text{LayerNorm}(\mathbf{x}) = \gamma \frac{\mathbf{x} - \mu_x}{\sigma_x} + \beta$$

## 7.4 Residual Connections (Skip Connections)

The **ResNet** breakthrough (He et al., 2015): instead of learning $\mathbf{h} = F(\mathbf{x})$, learn the **residual** $\mathbf{h} = F(\mathbf{x}) + \mathbf{x}$:

$$\mathbf{y} = F(\mathbf{x}) + \mathbf{x}$$

**Why this matters for gradients:**
$$\frac{\partial \mathcal{L}}{\partial \mathbf{x}} = \frac{\partial \mathcal{L}}{\partial \mathbf{y}} \left( \frac{\partial F(\mathbf{x})}{\partial \mathbf{x}} + 1 \right)$$

The `+1` means there's always a direct gradient highway — even if $F$'s gradients are tiny, the gradient still flows! This is why residual connections are **mandatory** in every Transformer.

**The residual stream** — in mechanistic interpretability, the residual connections form the "residual stream" — a communication channel that every layer reads from and writes to. Understanding this is fundamental to understanding transformers mechanistically.

## 7.5 Dropout — Regularization by Noise

During training, randomly zero out activations with probability $p$:

$$h_i \leftarrow \begin{cases} h_i / (1-p) & \text{with probability } (1-p) \\ 0 & \text{with probability } p \end{cases}$$

(The $1/(1-p)$ scaling keeps expected values correct)

**Why it works:** Forces the network to learn redundant representations. No neuron can rely entirely on any single other neuron. This improves generalization dramatically.

In [ ]:
# =============================================================
# 7.1-7.5 — Deep Networks with PyTorch: BatchNorm, Dropout, Residuals
# =============================================================

import torch
import torch.nn as nn

# --- Layer Norm from scratch ---
class LayerNormManual(nn.Module):
    """LayerNorm implemented from scratch for understanding."""
    def __init__(self, d_model, eps=1e-6):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(d_model))  # Learned scale
        self.beta  = nn.Parameter(torch.zeros(d_model)) # Learned shift
        self.eps = eps
    
    def forward(self, x):
        # Normalize over last dimension (feature dim)
        mean = x.mean(dim=-1, keepdim=True)          # (B, T, 1)
        var  = x.var(dim=-1, keepdim=True, unbiased=False)  # (B, T, 1)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)    # (B, T, D)
        return self.gamma * x_norm + self.beta        # (B, T, D)

# --- Residual Block from scratch ---
class ResidualBlock(nn.Module):
    """Residual block: y = F(x) + x"""
    def __init__(self, dim, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 4),  # Expand
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 4, dim),  # Project back
            nn.Dropout(dropout),
        )
        self.norm = nn.LayerNorm(dim)
    
    def forward(self, x):
        # Pre-norm architecture (common in modern transformers)
        return x + self.net(self.norm(x))  # x + F(LayerNorm(x))

# --- Deep MLP with all modern tricks ---
class DeepMLP(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, n_layers, dropout=0.1):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.ModuleList([
            ResidualBlock(hidden_dim, dropout) for _ in range(n_layers)
        ])
        self.output_proj = nn.Linear(hidden_dim, output_dim)
    
    def forward(self, x):
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)          # Each block: x + F(LayerNorm(x))
        return self.output_proj(x)

# --- Compare: With vs Without Residuals ---
print("=== Gradient Flow with vs without Residual Connections ===")
print()

# Simulate gradient norms at each layer depth
np.random.seed(42)
n_layers = 20
dim = 64

x = torch.randn(16, dim, requires_grad=False)

# Without residuals (plain deep network)
class PlainBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.Tanh())
    def forward(self, x): return self.net(x)

# With residuals
class ResBlock(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim), nn.ReLU())
    def forward(self, x): return x + self.net(x)  # Residual!

grad_norms_plain = []
grad_norms_res   = []

for n_l in range(1, n_layers + 1):
    torch.manual_seed(0)
    # Plain
    plain = nn.Sequential(*[PlainBlock(dim) for _ in range(n_l)])
    x_in = torch.randn(4, dim, requires_grad=True)
    loss = plain(x_in).sum()
    loss.backward()
    grad_norms_plain.append(x_in.grad.norm().item())
    
    # Residual
    torch.manual_seed(0)
    res = nn.Sequential(*[ResBlock(dim) for _ in range(n_l)])
    x_in2 = torch.randn(4, dim, requires_grad=True)
    loss2 = res(x_in2).sum()
    loss2.backward()
    grad_norms_res.append(x_in2.grad.norm().item())

plt.figure(figsize=(12, 5))
plt.subplot(1, 2, 1)
plt.plot(range(1, n_layers+1), grad_norms_plain, 'r-o', label='Plain (Tanh)', markersize=4)
plt.plot(range(1, n_layers+1), grad_norms_res, 'b-o', label='Residual (ReLU)', markersize=4)
plt.xlabel('Number of Layers'); plt.ylabel('Input Gradient Norm')
plt.title('Gradient Flow: Plain vs Residual Networks', fontweight='bold')
plt.legend(); plt.yscale('log')

plt.subplot(1, 2, 2)
# Illustrate residual block
ax = plt.gca()
ax.set_xlim(0, 4); ax.set_ylim(0, 6)
ax.axis('off')

# Draw residual block diagram
boxes = [(1, 4.5, 'Input x'), (1, 3, 'LayerNorm'), (1, 1.5, 'FFN + GELU')]
for bx, by, label in boxes:
    ax.add_patch(plt.Rectangle((bx-0.7, by-0.35), 1.4, 0.7,
                                facecolor='#3498db', edgecolor='black', linewidth=2, alpha=0.7))
    ax.text(bx, by, label, ha='center', va='center', fontsize=10, fontweight='bold', color='white')

ax.add_patch(plt.Rectangle((1-0.4, 0.15), 0.8, 0.7,
                             facecolor='#2ecc71', edgecolor='black', linewidth=2, alpha=0.8))
ax.text(1, 0.5, 'Output\nx + F(x)', ha='center', va='center', fontsize=9, fontweight='bold')

# Arrows for main path
for y1, y2 in [(4.15, 3.35), (2.65, 1.85), (1.15, 0.85)]:
    ax.annotate('', xy=(1, y2), xytext=(1, y1),
                arrowprops=dict(arrowstyle='->', lw=2, color='black'))

# Residual/skip connection (curved arrow)
ax.annotate('', xy=(1.7, 0.5), xytext=(1.7, 4.5),
            arrowprops=dict(arrowstyle='->', lw=2.5, color='#e74c3c',
                            connectionstyle='arc3,rad=0.4'))
ax.text(2.5, 2.5, 'Skip\nconnection\n(gradient highway!)', color='#e74c3c',
        ha='center', va='center', fontsize=9, fontweight='bold')

ax.set_title('Residual Block Architecture', fontweight='bold')

plt.tight_layout()
plt.show()

---
# Section 8: Sequences & Recurrence — Before Transformers
---

Language is sequential. You can't process a sentence by treating each word independently — context matters. Before transformers, the dominant approach was **Recurrent Neural Networks (RNNs)**.

## 8.1 Recurrent Neural Networks

An RNN processes a sequence step-by-step, maintaining a **hidden state** $\mathbf{h}_t$ that summarizes all context seen so far:

$$\mathbf{h}_t = \phi(\mathbf{W}_h \mathbf{h}_{t-1} + \mathbf{W}_x \mathbf{x}_t + \mathbf{b})$$

$$\mathbf{y}_t = \mathbf{W}_o \mathbf{h}_t + \mathbf{b}_o$$

**Problem:** Backpropagation Through Time (BPTT) requires multiplying $\mathbf{W}_h$ many times — gradients vanish or explode over long sequences. An RNN with 1000 tokens has 1000 gradient multiplications.

## 8.2 Long Short-Term Memory (LSTM)

LSTMs solve the vanishing gradient problem with a **cell state** $\mathbf{c}_t$ and **gates** that control information flow:

$$\mathbf{f}_t = \sigma(\mathbf{W}_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate: what to erase)}$$
$$\mathbf{i}_t = \sigma(\mathbf{W}_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate: what to write)}$$
$$\tilde{\mathbf{c}}_t = \tanh(\mathbf{W}_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(new candidate content)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(update cell state)}$$
$$\mathbf{o}_t = \sigma(\mathbf{W}_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t)$$

**The key insight:** The cell state $\mathbf{c}_t$ flows with only element-wise multiplications — gradients can flow much more easily across long sequences. The gates learn what to remember and what to forget.

**Why we use transformers now instead:** RNNs/LSTMs process tokens sequentially — you must finish token $t$ before processing $t+1$. This makes training slow and hard to parallelize. Transformers process **all tokens simultaneously** via attention. With modern GPUs that excel at parallel computation, this is a massive win.

In [ ]:
# =============================================================
# 8.1-8.2 — RNN and LSTM from Scratch
# =============================================================

class RNNCell:
    """A single RNN cell, implemented from scratch in NumPy."""
    def __init__(self, input_dim, hidden_dim):
        scale = 0.01
        self.Wh = np.random.randn(hidden_dim, hidden_dim) * scale
        self.Wx = np.random.randn(input_dim, hidden_dim)  * scale
        self.b  = np.zeros(hidden_dim)
    
    def step(self, x, h_prev):
        """One step: h_t = tanh(W_h @ h_{t-1} + W_x @ x_t + b)"""
        return np.tanh(h_prev @ self.Wh + x @ self.Wx + self.b)
    
    def forward(self, X_seq):
        """Process a full sequence. X_seq: (T, input_dim)"""
        T, _ = X_seq.shape
        h = np.zeros(self.Wh.shape[0])
        hidden_states = []
        for t in range(T):
            h = self.step(X_seq[t], h)
            hidden_states.append(h.copy())
        return np.array(hidden_states)  # (T, hidden_dim)

# Simulate: does the hidden state remember the beginning of a sequence?
np.random.seed(42)
rnn = RNNCell(input_dim=4, hidden_dim=16)

T = 30  # 30-step sequence
X_seq = np.random.randn(T, 4)
hidden_states = rnn.forward(X_seq)

print("=== RNN Hidden State Analysis ===")
print(f"Sequence length: {T} steps")
print(f"Hidden state shape at each step: {hidden_states[0].shape}")

# Show hidden state norms over time
norms = np.linalg.norm(hidden_states, axis=1)
print(f"Hidden state norm at t=1:  {norms[0]:.4f}")
print(f"Hidden state norm at t=15: {norms[14]:.4f}")
print(f"Hidden state norm at t=30: {norms[29]:.4f}")

# Visualize with heatmap
fig, axes = plt.subplots(1, 2, figsize=(16, 4))

im = axes[0].imshow(hidden_states.T, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Time step t'); axes[0].set_ylabel('Hidden dimension')
axes[0].set_title('RNN Hidden States Over Time (16 neurons × 30 steps)', fontweight='bold')

# Demonstrate vanishing gradients numerically
# Gradient of h_T with respect to h_0 involves T matrix multiplications
def gradient_norm_through_rnn(Wh, T):
    """Approximate gradient norm after T steps (via Jacobian product)."""
    J = np.eye(Wh.shape[0])
    scale_factor = 0.5  # tanh derivative ≤ 1, roughly 0.5 on average
    for t in range(T):
        J = J @ (Wh * scale_factor)  # Chain rule: multiply by Wh * tanh'
    return np.linalg.norm(J, 'fro')

Wh_stable = np.random.randn(16, 16) * 0.3   # Small weights → vanishing
Wh_large  = np.random.randn(16, 16) * 1.5   # Large weights → exploding

steps = range(1, 51)
norms_stable = [gradient_norm_through_rnn(Wh_stable, t) for t in steps]
norms_large  = [gradient_norm_through_rnn(Wh_large, t) for t in steps]

axes[1].semilogy(steps, norms_stable, '#3498db', linewidth=2, label='Small Wh (vanishing)')
axes[1].semilogy(steps, norms_large, '#e74c3c', linewidth=2, label='Large Wh (exploding)')
axes[1].axhline(1, color='k', linestyle='--', alpha=0.5, label='Stable')
axes[1].set_xlabel('Sequence Length T'); axes[1].set_ylabel('Gradient Norm (log scale)')
axes[1].set_title('Vanishing & Exploding Gradients in RNNs', fontweight='bold')
axes[1].legend()

plt.tight_layout()
plt.show()

print("\n💡 This is why transformers replaced RNNs:")
print("   Transformers connect ANY two positions directly via attention")
print("   → No gradient through T sequential multiplications!")

---
# Section 9: The Attention Mechanism — The Core of Transformers
---

The attention mechanism is the most important innovation in the history of modern AI. Everything else — GPT, Llama, BERT, Claude — is built on top of it.

## 9.1 The Intuition

When you read the sentence "The **animal** didn't cross the street because **it** was too tired," you immediately know "it" refers to "animal," not "street." You did this by **attending** to the relevant word.

Attention lets each token in a sequence directly look at and gather information from any other token — regardless of distance. No sequential processing needed.

## 9.2 Queries, Keys, and Values

The attention mechanism uses three projections of the input:

- **Query** $\mathbf{Q}$: "What am I looking for?" — what information does this position want?
- **Key** $\mathbf{K}$: "What do I contain?" — what does each position offer?
- **Value** $\mathbf{V}$: "What do I pass on?" — the actual content to aggregate

$$\mathbf{Q} = \mathbf{X}\mathbf{W}_Q, \quad \mathbf{K} = \mathbf{X}\mathbf{W}_K, \quad \mathbf{V} = \mathbf{X}\mathbf{W}_V$$

## 9.3 Scaled Dot-Product Attention

$$\text{Attention}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = \text{softmax}\left(\frac{\mathbf{Q}\mathbf{K}^T}{\sqrt{d_k}}\right) \mathbf{V}$$

**Step by step:**

1. **$\mathbf{Q}\mathbf{K}^T$** — compute similarity scores between every pair of positions. Shape: $(T, T)$ — each token queries every other token.

2. **$/ \sqrt{d_k}$** — scale down to prevent softmax saturation. If $d_k$ is large, dot products grow large, pushing softmax to near-argmax and killing gradients.

3. **$\text{softmax}(...)$** — normalize scores to probabilities. Now each row sums to 1 — these are the **attention weights**, telling each position how much to attend to each other position.

4. **$\cdot \mathbf{V}$** — weighted sum of values. Each position gets a mixture of values, weighted by how much attention it pays to each other position.

**Causal masking (for language models):** When generating text, position $t$ should only attend to positions $\leq t$ (can't see the future!). We add $-\infty$ to future positions before softmax, making their attention weight 0.

## 9.4 Multi-Head Attention

Instead of one attention operation, run $h$ attention operations in parallel with different learned projections:

$$\text{head}_i = \text{Attention}(\mathbf{Q}\mathbf{W}_i^Q, \mathbf{K}\mathbf{W}_i^K, \mathbf{V}\mathbf{W}_i^V)$$
$$\text{MultiHead}(\mathbf{Q}, \mathbf{K}, \mathbf{V}) = [\text{head}_1, ..., \text{head}_h]\mathbf{W}^O$$

**Why multiple heads?** Different heads learn to attend to different types of relationships simultaneously — one head might track subject-verb agreement, another might track coreference ("it" → "animal"), another might track syntactic structure. The model discovers these patterns from data.

In [ ]:
# =============================================================
# 9.3-9.4 — Attention Mechanism: Complete Implementation & Visualization
# =============================================================

def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Scaled dot-product attention from scratch.
    
    Args:
        Q: (batch, heads, seq_q, d_k)  — queries
        K: (batch, heads, seq_k, d_k)  — keys
        V: (batch, heads, seq_v, d_v)  — values  (seq_k == seq_v)
        mask: (seq_q, seq_k) or None   — causal mask (True = mask out)
    
    Returns:
        output: (batch, heads, seq_q, d_v)
        attn_weights: (batch, heads, seq_q, seq_k)
    """
    d_k = Q.shape[-1]
    
    # Step 1: QK^T — similarity scores
    scores = torch.matmul(Q, K.transpose(-2, -1))  # (B, H, T_q, T_k)
    
    # Step 2: Scale by sqrt(d_k)
    scores = scores / (d_k ** 0.5)
    
    # Step 3: Apply causal mask (for autoregressive LMs)
    if mask is not None:
        scores = scores.masked_fill(mask, float('-inf'))
    
    # Step 4: Softmax over key dimension
    attn_weights = torch.softmax(scores, dim=-1)  # (B, H, T_q, T_k)
    
    # Step 5: Weighted sum of values
    output = torch.matmul(attn_weights, V)         # (B, H, T_q, d_v)
    
    return output, attn_weights


class MultiHeadAttention(nn.Module):
    """
    Multi-head attention: run n_heads attention operations in parallel,
    each looking at a different subspace.
    """
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0, "d_model must be divisible by n_heads"
        
        self.d_model  = d_model
        self.n_heads  = n_heads
        self.d_k      = d_model // n_heads  # Dimension per head
        
        # Weight matrices: project to Q, K, V spaces
        self.W_Q = nn.Linear(d_model, d_model, bias=False)  # (d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)  # Output projection
    
    def split_heads(self, x):
        """Reshape (B, T, d_model) → (B, n_heads, T, d_k)"""
        B, T, _ = x.shape
        x = x.view(B, T, self.n_heads, self.d_k)  # (B, T, H, d_k)
        return x.transpose(1, 2)                    # (B, H, T, d_k)
    
    def merge_heads(self, x):
        """Reshape (B, n_heads, T, d_k) → (B, T, d_model)"""
        B, H, T, d_k = x.shape
        x = x.transpose(1, 2)          # (B, T, H, d_k)
        return x.contiguous().view(B, T, H * d_k)  # (B, T, d_model)
    
    def forward(self, x, mask=None):
        """
        x: (B, T, d_model)  — the sequence
        Returns: (B, T, d_model)
        """
        # Project to Q, K, V
        Q = self.split_heads(self.W_Q(x))  # (B, H, T, d_k)
        K = self.split_heads(self.W_K(x))  # (B, H, T, d_k)
        V = self.split_heads(self.W_V(x))  # (B, H, T, d_k)
        
        # Attention
        attn_output, attn_weights = scaled_dot_product_attention(Q, K, V, mask)
        
        # Merge heads and project back
        output = self.W_O(self.merge_heads(attn_output))  # (B, T, d_model)
        
        return output, attn_weights


# --- Test and Visualize ---
torch.manual_seed(42)
B, T, d_model, n_heads = 1, 8, 64, 4

# Simulate a sequence of 8 tokens (e.g., "The cat sat on the mat today .")
tokens = ["The", "cat", "sat", "on", "the", "mat", "today", "."]
x = torch.randn(B, T, d_model)

# Causal mask: lower triangular = can see, upper = cannot
causal_mask = torch.triu(torch.ones(T, T), diagonal=1).bool()

mha = MultiHeadAttention(d_model, n_heads)
output, attn_weights = mha(x, mask=causal_mask)

print(f"Input:   {x.shape}")
print(f"Output:  {output.shape}  (same shape — attention is a transformation, not size change)")
print(f"Weights: {attn_weights.shape}  (batch=1, heads=4, query_pos=8, key_pos=8)")

# Visualize attention patterns for each head
fig, axes = plt.subplots(1, n_heads, figsize=(18, 4))

attn_np = attn_weights.detach().squeeze(0).numpy()  # (n_heads, T, T)

for h in range(n_heads):
    ax = axes[h]
    im = ax.imshow(attn_np[h], cmap='Blues', vmin=0, vmax=1, aspect='auto')
    ax.set_xticks(range(T)); ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=9)
    ax.set_yticks(range(T)); ax.set_yticklabels(tokens, fontsize=9)
    ax.set_title(f'Head {h+1}', fontweight='bold')
    ax.set_xlabel('Keys (attended to)')
    if h == 0: ax.set_ylabel('Queries (attending from)')
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

plt.suptitle('Multi-Head Attention Weights (with Causal Mask)\nDark = High attention, Upper triangle = -∞ (masked)',
             fontweight='bold', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

print("\n💡 Notice the upper-right triangle is always 0 (masked)")
print("   Token 't' can only attend to tokens 1...t — this enforces causality")
print("   Different heads learn different attention patterns (when trained)")

---
# Section 10: The Full Transformer Architecture
---

The Transformer (Vaswani et al., "Attention is All You Need", 2017) is the architecture that changed everything. Every modern LLM — GPT, Llama, Claude, Gemini — is a Transformer.

## 10.1 Architecture Overview

A Transformer consists of stacked **Transformer blocks**, each containing:

1. **Multi-Head Self-Attention** — tokens attend to each other
2. **Layer Normalization** — stabilize activations
3. **Feed-Forward Network (FFN)** — process each token independently
4. **Residual connections** — gradient highways around each component

**GPT-style (decoder-only) block:**
```
x → LayerNorm → MaskedSelfAttention → + x  →  LayerNorm → FFN → + x
         ↑_________________________________↑   ↑_____________________↑
              (residual connection)                (residual connection)
```

## 10.2 Positional Encoding

Attention is **permutation-invariant** — it doesn't care about order by default. Position information must be injected explicitly.

**Sinusoidal positional encoding (original Transformer):**
$$PE_{(pos, 2i)} = \sin\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$
$$PE_{(pos, 2i+1)} = \cos\left(\frac{pos}{10000^{2i/d_{model}}}\right)$$

**Learned positional embeddings (GPT-2, most modern LLMs):** Simply learn a lookup table: one embedding vector per position. Simple and effective.

**Rotary Position Embeddings (RoPE)** — used in LLaMA, Mistral, GPT-NeoX: Encode position by rotating Q and K vectors. Naturally handles relative positions and generalizes to lengths beyond training.

## 10.3 The Feed-Forward Network (FFN)

Each position is processed independently by a 2-layer MLP:

$$\text{FFN}(\mathbf{x}) = \mathbf{W}_2 \cdot \text{GELU}(\mathbf{W}_1 \mathbf{x} + \mathbf{b}_1) + \mathbf{b}_2$$

Typically, the hidden dimension is **4× the model dimension**: if $d_{model} = 768$, then $d_{ff} = 3072$.

**The FFN stores facts.** Research has shown that factual associations ("The Eiffel Tower is in ___") are stored in the FFN weights, while attention handles information routing. This is key intuition for mechanistic interpretability.

## 10.4 The Residual Stream — A Researcher's Perspective

Every Transformer layer reads from the **residual stream** and writes back to it:

$$\mathbf{x}^{(l+1)} = \mathbf{x}^{(l)} + \text{Attn}^{(l)}(\mathbf{x}^{(l)}) + \text{FFN}^{(l)}(\mathbf{x}^{(l)})$$

The residual stream is not just a gradient trick — it's the **central data structure** of the transformer. Neel Nanda's framework for mechanistic interpretability views the residual stream as the "global workspace" where all components communicate, each making small contributions.

In [ ]:
# =============================================================
# 10 — Complete Transformer Block Implementation (GPT-style)
# =============================================================

class PositionalEncoding(nn.Module):
    """Sinusoidal positional encoding (original Transformer paper)."""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        # Create the PE matrix once
        PE = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()          # (max_len, 1)
        div = torch.exp(torch.arange(0, d_model, 2).float() *     # (d_model/2,)
                        -(np.log(10000.0) / d_model))
        PE[:, 0::2] = torch.sin(pos * div)  # Even dimensions
        PE[:, 1::2] = torch.cos(pos * div)  # Odd dimensions
        self.register_buffer('PE', PE.unsqueeze(0))  # (1, max_len, d_model)
    
    def forward(self, x):
        return x + self.PE[:, :x.size(1)]  # Add positional encoding


class FeedForward(nn.Module):
    """Position-wise Feed-Forward Network: expand → activate → project."""
    def __init__(self, d_model, d_ff=None, dropout=0.1):
        super().__init__()
        d_ff = d_ff or 4 * d_model  # Standard: 4× expansion
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),   # (d_model → 4*d_model)
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),   # (4*d_model → d_model)
            nn.Dropout(dropout),
        )
    
    def forward(self, x):
        return self.net(x)


class TransformerBlock(nn.Module):
    """
    GPT-style Transformer block (decoder-only, pre-norm).
    
    Each block:
      x → LayerNorm → MaskedSelfAttention → + residual
        → LayerNorm → FFN → + residual
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, n_heads)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = FeedForward(d_model, d_ff, dropout)
    
    def forward(self, x, mask=None):
        # Pre-norm + attention + residual
        attn_out, attn_weights = self.attn(self.ln1(x), mask=mask)
        x = x + attn_out                  # Residual: "write to stream"
        
        # Pre-norm + FFN + residual
        x = x + self.ffn(self.ln2(x))     # Residual: "write to stream"
        
        return x, attn_weights


class MiniGPT(nn.Module):
    """
    A small GPT-style decoder-only Transformer.
    Shows the full architecture: embedding → positional encoding → N blocks → output.
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_seq_len, dropout=0.1):
        super().__init__()
        
        self.d_model = d_model
        self.max_seq_len = max_seq_len
        
        # Token embedding
        self.token_emb = nn.Embedding(vocab_size, d_model)
        # Position embedding (learned, GPT-style)
        self.pos_emb   = nn.Embedding(max_seq_len, d_model)
        self.dropout   = nn.Dropout(dropout)
        
        # Transformer blocks (the core)
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, dropout=dropout)
            for _ in range(n_layers)
        ])
        
        # Final layer norm + language model head
        self.ln_final = nn.LayerNorm(d_model)
        self.lm_head  = nn.Linear(d_model, vocab_size, bias=False)
        
        # Weight tying: embedding and lm_head share weights (common in modern LLMs)
        self.lm_head.weight = self.token_emb.weight
        
        # Count parameters
        n_params = sum(p.numel() for p in self.parameters())
        print(f"MiniGPT initialized: {n_params:,} parameters")
        print(f"  Architecture: vocab={vocab_size}, d_model={d_model}, "
              f"n_heads={n_heads}, n_layers={n_layers}")
    
    def forward(self, idx, targets=None):
        """
        idx: (B, T) — token indices
        Returns: logits (B, T, vocab_size) and optionally loss
        """
        B, T = idx.shape
        assert T <= self.max_seq_len, f"Sequence too long: {T} > {self.max_seq_len}"
        
        # 1. Token + Position embeddings
        positions = torch.arange(T, device=idx.device)              # (T,)
        x = self.token_emb(idx) + self.pos_emb(positions)          # (B, T, d_model)
        x = self.dropout(x)
        
        # 2. Causal mask: token at position t can only see positions 0..t
        mask = torch.triu(torch.ones(T, T, device=idx.device), diagonal=1).bool()
        
        # 3. Pass through transformer blocks
        all_attn_weights = []
        for block in self.blocks:
            x, attn_w = block(x, mask=mask)
            all_attn_weights.append(attn_w)
        
        # 4. Final layer norm
        x = self.ln_final(x)  # (B, T, d_model)
        
        # 5. LM head: project to vocab logits
        logits = self.lm_head(x)  # (B, T, vocab_size)
        
        # 6. Compute loss if targets provided
        loss = None
        if targets is not None:
            # Cross-entropy: predict next token
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # (B*T, vocab_size)
                targets.view(-1)                   # (B*T,)
            )
        
        return logits, loss, all_attn_weights
    
    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0):
        """Autoregressive generation: sample one token at a time."""
        for _ in range(max_new_tokens):
            # Crop to max_seq_len if needed
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _, _ = self.forward(idx_cond)
            # Take logits at last position only
            logits = logits[:, -1, :] / temperature  # (B, vocab_size)
            probs = F.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, next_token], dim=1)
        return idx


# --- Instantiate and inspect ---
torch.manual_seed(42)
model = MiniGPT(
    vocab_size=1000,
    d_model=128,
    n_heads=4,
    n_layers=4,
    max_seq_len=64
)

# Forward pass with dummy data
batch_size, seq_len = 2, 16
dummy_input   = torch.randint(0, 1000, (batch_size, seq_len))
dummy_targets = torch.randint(0, 1000, (batch_size, seq_len))

logits, loss, attn_weights_list = model(dummy_input, dummy_targets)

print(f"\nInput shape:   {dummy_input.shape}")
print(f"Logits shape:  {logits.shape}  — (batch, seq, vocab)")
print(f"Loss:          {loss.item():.4f}  (cross-entropy, random ≈ log(1000) = {np.log(1000):.2f})")
print(f"Attn shapes:   {attn_weights_list[0].shape}  — (batch, heads, seq, seq) per layer")

In [ ]:
# =============================================================
# 10.2 — Positional Encoding Visualization
# =============================================================

# Compute sinusoidal PE
max_len, d_model = 100, 64
PE = np.zeros((max_len, d_model))
pos = np.arange(max_len)[:, np.newaxis]
div = np.exp(np.arange(0, d_model, 2) * -(np.log(10000.0) / d_model))
PE[:, 0::2] = np.sin(pos * div)
PE[:, 1::2] = np.cos(pos * div)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Full PE matrix
im = axes[0].imshow(PE, aspect='auto', cmap='RdBu', vmin=-1, vmax=1)
plt.colorbar(im, ax=axes[0])
axes[0].set_xlabel('Embedding dimension'); axes[0].set_ylabel('Position')
axes[0].set_title('Sinusoidal Positional Encoding Matrix\n(each row = position embedding)', fontweight='bold')

# A few dimensions over position
for dim, color in [(0, '#e74c3c'), (2, '#3498db'), (10, '#2ecc71'), (20, '#9b59b6')]:
    axes[1].plot(PE[:, dim], color=color, linewidth=2, label=f'dim {dim}')
axes[1].set_xlabel('Position'); axes[1].set_ylabel('Value')
axes[1].set_title('PE Values Across Positions\n(lower dim = lower frequency)', fontweight='bold')
axes[1].legend()

# Similarity between position encodings (dot product)
PE_sim = PE @ PE.T  # (100, 100)
im2 = axes[2].imshow(PE_sim, cmap='RdBu', aspect='auto')
plt.colorbar(im2, ax=axes[2])
axes[2].set_xlabel('Position'); axes[2].set_ylabel('Position')
axes[2].set_title('PE Similarity Matrix (PE @ PE.T)\nNearby positions are similar', fontweight='bold')

plt.tight_layout()
plt.show()

print("\n💡 Key property: PE(pos) · PE(pos + k) depends only on k, not pos")
print("   This means the model can learn relative position relationships")
print("   RoPE (used in LLaMA) extends this idea even further")

---
# Section 11: Language Models & LLMs
---

## 11.1 What is a Language Model?

A language model assigns a probability to sequences of tokens:

$$P(w_1, w_2, ..., w_T) = \prod_{t=1}^{T} P(w_t | w_1, ..., w_{t-1})$$

Training objective: **Next-token prediction** (autoregressive modeling). Given a sequence, predict the next token. Loss:

$$\mathcal{L} = -\frac{1}{T} \sum_{t=1}^{T} \log P(w_t | w_1, ..., w_{t-1}; \theta)$$

This single objective — predict the next word — done at massive scale on massive data, produces a model that can reason, translate, code, write, and more. It's stunning how much emerges from this simple objective.

## 11.2 Tokenization — Breaking Text into Numbers

Neural networks work on numbers, not text. We need to convert text → integer token IDs.

**Byte Pair Encoding (BPE)** — used in GPT-2, GPT-4, LLaMA:
1. Start with individual characters as tokens
2. Repeatedly merge the most frequent adjacent pair
3. Stop when vocabulary is the right size (50K–100K for modern LLMs)

**Key properties:**
- Common words → single token ("the" → token 262)
- Rare words → multiple tokens ("unbelievably" → ["un", "beli", "ev", "ably"])
- The vocabulary has meaning: token 262 always means "the"

## 11.3 Perplexity — Measuring Language Model Quality

Perplexity is the standard metric for language model quality:

$$\text{PPL} = e^{\mathcal{L}} = e^{-\frac{1}{T}\sum_t \log P(w_t | \text{context})}$$

- **Intuition:** "How many tokens is the model choosing between, on average?" Lower = better.
- A random model over 50K vocab: PPL = 50,000
- GPT-2: ~40 on WikiText-103
- GPT-4: estimated ~5-10 on benchmarks

## 11.4 Scaling Laws — The Key to Understanding LLMs

Kaplan et al. (2020) from OpenAI discovered that LLM performance follows **power laws**:

$$\mathcal{L}(N) \approx \left(\frac{N_c}{N}\right)^{\alpha_N} \quad \text{(more parameters → lower loss)}$$
$$\mathcal{L}(D) \approx \left(\frac{D_c}{D}\right)^{\alpha_D} \quad \text{(more data → lower loss)}$$

**Chinchilla scaling laws** (Hoffmann et al., 2022): Optimal compute-matching says use roughly **20 tokens per parameter** — LLaMA followed this.

**Emergent capabilities:** At certain scale thresholds, capabilities appear suddenly — chain-of-thought reasoning, few-shot learning, in-context learning. These weren't explicitly trained for.

## 11.5 The LLM Training Pipeline

Modern LLMs go through multiple stages:

1. **Pre-training:** Next-token prediction on massive web corpus (terabytes of text). This is the expensive part.

2. **Supervised Fine-Tuning (SFT):** Train on high-quality (prompt, response) pairs. Teaches the model to follow instructions.

3. **Reinforcement Learning from Human Feedback (RLHF):** Humans compare responses → train reward model → use RL (PPO) to maximize reward while staying close to SFT model (via KL penalty).

4. **Constitutional AI / RLAIF (Anthropic):** Use AI feedback instead of or alongside human feedback.

In [ ]:
# =============================================================
# 11 — Training a Tiny Character-Level Language Model on Shakespeare
# (Inspired by Karpathy's makemore — a pedagogical classic)
# =============================================================

# --- Load some text data ---
tiny_shakespeare = """To be, or not to be, that is the question:
Whether 'tis nobler in the mind to suffer
The slings and arrows of outrageous fortune,
Or to take arms against a sea of troubles
And by opposing end them. To die—to sleep,
No more; and by a sleep to say we end
The heart-ache and the thousand natural shocks
That flesh is heir to: 'tis a consummation
Devoutly to be wish'd. To die, to sleep;
To sleep, perchance to dream—ay, there's the rub:
For in that sleep of death what dreams may come,
When we have shuffled off this mortal coil,
Must give us pause—there's the respect
That makes calamity of so long life."""

# Character-level tokenization
chars = sorted(list(set(tiny_shakespeare)))
vocab_size = len(chars)
char_to_idx = {ch: i for i, ch in enumerate(chars)}
idx_to_char = {i: ch for i, ch in enumerate(chars)}

encode = lambda s: [char_to_idx[c] for c in s]
decode = lambda l: ''.join([idx_to_char[i] for i in l])

data = torch.tensor(encode(tiny_shakespeare), dtype=torch.long)

print(f"Dataset: {len(tiny_shakespeare)} characters")
print(f"Vocabulary: {vocab_size} unique characters: {chars[:20]}...")
print(f"Encoded (first 20): {data[:20].tolist()}")
print(f"Decoded back: '{decode(data[:20].tolist())}'")

# --- Create a tiny GPT ---
torch.manual_seed(42)
block_size = 32  # Context length
d_model = 64; n_heads = 4; n_layers = 3

tiny_gpt = MiniGPT(
    vocab_size=vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    n_layers=n_layers,
    max_seq_len=block_size
)

optimizer = torch.optim.AdamW(tiny_gpt.parameters(), lr=3e-3, weight_decay=0.01)

def get_batch(data, block_size, batch_size=32):
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x  = torch.stack([data[i:i+block_size]   for i in ix])
    y  = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

# --- Training ---
print("\nTraining...")
n_steps = 1000
losses = []

for step in range(n_steps):
    xb, yb = get_batch(data, block_size)
    _, loss, _ = tiny_gpt(xb, yb)
    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(tiny_gpt.parameters(), 1.0)  # Gradient clipping
    optimizer.step()
    losses.append(loss.item())
    
    if (step + 1) % 200 == 0:
        print(f"  Step {step+1:4d}/{n_steps} | Loss: {loss.item():.4f} | PPL: {np.exp(loss.item()):.2f}")

# --- Generate text ---
print("\nGenerating text from trained model:")
print("-" * 50)
tiny_gpt.eval()
context = torch.tensor([[char_to_idx['T']]], dtype=torch.long)
generated = tiny_gpt.generate(context, max_new_tokens=200, temperature=0.8)
print(decode(generated[0].tolist()))
print("-" * 50)

# --- Training curve ---
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
smoothed = np.convolve(losses, np.ones(20)/20, mode='valid')
plt.plot(losses, alpha=0.3, color='blue', label='Raw loss')
plt.plot(smoothed, color='blue', linewidth=2, label='Smoothed loss')
plt.xlabel('Training Step'); plt.ylabel('Cross-Entropy Loss')
plt.title('Character-Level LM Training', fontweight='bold'); plt.legend()

plt.subplot(1, 2, 2)
perplexities = [np.exp(l) for l in losses]
plt.plot(np.convolve(perplexities, np.ones(20)/20, mode='valid'), color='red', linewidth=2)
plt.xlabel('Training Step'); plt.ylabel('Perplexity')
plt.title('Perplexity = exp(Loss)', fontweight='bold')
plt.tight_layout()
plt.show()

print(f"\nFinal loss: {losses[-1]:.4f} | Final PPL: {np.exp(losses[-1]):.2f}")
print(f"(PPL ≈ {np.exp(losses[-1]):.0f} means the model is choosing between ~{np.exp(losses[-1]):.0f} chars on average)")

In [ ]:
# =============================================================
# 11.4 — Scaling Laws Visualization
# =============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Simulate scaling law: L(N) ≈ (N_c/N)^alpha_N
N_params = np.logspace(7, 13, 100)  # 10M to 10B params
N_c = 1e13; alpha_N = 0.076
loss_vs_params = (N_c / N_params) ** alpha_N

# Known model data points (approximate)
models = {
    'GPT-2 117M': (1.17e8, 3.0),
    'GPT-2 1.5B': (1.5e9, 2.7),
    'GPT-3 13B': (1.3e10, 2.1),
    'GPT-3 175B': (1.75e11, 1.8),
}

ax = axes[0]
ax.loglog(N_params, loss_vs_params, 'b-', linewidth=2.5, label='Scaling law: L ∝ N^(-α)')
for name, (N, L) in models.items():
    ax.loglog(N, L, 'ro', markersize=10)
    ax.annotate(name, (N, L), textcoords='offset points', xytext=(8, 4), fontsize=8)
ax.set_xlabel('Model Parameters N'); ax.set_ylabel('Validation Loss')
ax.set_title('Scaling Law: Loss vs Parameters', fontweight='bold')
ax.legend()

# Compute-optimal frontier (Chinchilla)
# For compute C, optimal N* and D* follow N* ∝ C^0.5, D* ∝ C^0.5
compute_flops = np.logspace(18, 26, 100)  # FLOPs
# Chinchilla: N_opt ≈ (C / 6*20)^0.5 roughly
N_opt = np.sqrt(compute_flops / (6 * 20))
D_opt = 20 * N_opt  # ~20 tokens per parameter

ax2 = axes[1]
ax2.loglog(N_opt, D_opt, 'g-', linewidth=2.5, label='Chinchilla optimal: D ≈ 20N')

# Mark known models
chinchilla_models = {
    'GPT-3 (overtrained)': (1.75e11, 3e11),
    'LLaMA-1 7B': (7e9, 1e12),
    'Chinchilla 70B': (7e10, 1.4e12),
    'LLaMA-2 70B': (7e10, 2e12),
}
for name, (N, D) in chinchilla_models.items():
    color = '#e74c3c' if 'GPT-3' in name else '#3498db'
    ax2.loglog(N, D, 'o', markersize=10, color=color)
    ax2.annotate(name, (N, D), textcoords='offset points', xytext=(8, 4), fontsize=8)

ax2.set_xlabel('Model Parameters N'); ax2.set_ylabel('Training Tokens D')
ax2.set_title('Chinchilla Scaling:\nCompute-Optimal Training', fontweight='bold')
ax2.legend()

plt.tight_layout()
plt.show()

print("Key Chinchilla insight:")
print("  GPT-3 (175B params, 300B tokens) was UNDERTRAINED on data")
print("  Optimal: 175B params needs ~3.5T tokens")
print("  LLaMA-2 corrected this: smaller model, more tokens")

---
# Section 12: Research Foundations — Mechanistic Interpretability
---

This section bridges your foundational knowledge to the research frontier you're aiming for.

## 12.1 The Core Problem: What do Neural Networks Actually Compute?

We've built neural networks that work remarkably well. But we don't know *why* they work or *what* they've learned. This is the central problem that **mechanistic interpretability** (mech interp) tries to solve.

> **Mechanistic Interpretability:** Reverse-engineer neural networks from the bottom up to understand the algorithms and data structures they implement.

Key researchers: **Chris Olah** (Anthropic), **Neel Nanda** (DeepMind), and teams at Anthropic, Goodfire AI, Apollo Research.

## 12.2 Features and Circuits

**Features:** Directions in activation space that represent human-interpretable concepts. Olah et al. found that vision networks have neurons responding to specific visual features (curves, dog faces, etc.).

**Circuits:** Subgraphs of the network (sets of neurons/attention heads connected by weights) that implement specific algorithms. Example: "indirect object identification" circuit in GPT-2.

## 12.3 Superposition — Why Models Are Hard to Interpret

**The fundamental obstacle:** Models represent more features than they have dimensions.

If a model has $d$ dimensions but needs to represent $n >> d$ features, it can't give each feature its own dimension. Instead, it **superimposes** features:

$$\mathbf{x}_{\text{model}} \approx \sum_{i=1}^{n} f_i \mathbf{v}_i$$

Where $\mathbf{v}_i$ are not orthogonal — features share dimensions! This is why individual neurons are hard to interpret (they participate in many features simultaneously — **polysemanticity**).

**Why does the model do this?** Because most features are sparse — they're only active on a small fraction of inputs. The model can pack many sparse features into fewer dimensions with low interference.

## 12.4 Sparse Autoencoders (SAEs) — The Key Tool

SAEs are the dominant current technique for finding interpretable features in the superposition. The idea:

1. Take activations from a model layer
2. Train an autoencoder to reconstruct them but with a **sparse hidden layer** (L1 penalty on activations)
3. The sparse hidden layer learns to disentangle superimposed features into separate, interpretable directions

$$\mathbf{z} = \text{ReLU}(\mathbf{W}_e (\mathbf{x} - \mathbf{b}_{dec}) + \mathbf{b}_{enc})$$
$$\hat{\mathbf{x}} = \mathbf{W}_d \mathbf{z} + \mathbf{b}_{dec}$$
$$\mathcal{L} = ||\mathbf{x} - \hat{\mathbf{x}}||_2^2 + \lambda ||\mathbf{z}||_1$$

The L1 penalty forces most features to be 0 — only the active features for a given input should be non-zero.

In [ ]:
# =============================================================
# 12.3 — Superposition: A Minimal Demonstration
# Reproducing the core idea from Elhage et al., "Toy Models of Superposition"
# =============================================================

print("=" * 60)
print("Demonstrating Superposition in a Toy Model")
print("Elhage et al. 2022: 'Toy Models of Superposition'")
print("=" * 60)
print()
print("Setup: 5 features → compressed to 2D → reconstructed")
print("Features are sparse (active with probability p=0.1)")
print()

class ToyModelSuperposition(nn.Module):
    """
    Minimal model demonstrating superposition.
    Maps n_features → hidden_dim → n_features
    When hidden_dim < n_features, model must use superposition.
    """
    def __init__(self, n_features, hidden_dim):
        super().__init__()
        # W: maps features to hidden space (columns are feature directions)
        self.W = nn.Parameter(torch.randn(hidden_dim, n_features) * 0.1)
        self.b = nn.Parameter(torch.zeros(n_features))
    
    def forward(self, x):
        # Encode: x → hidden (linear + ReLU)
        h = F.relu(self.W @ x.unsqueeze(-1)).squeeze(-1) if x.dim() == 1 else \
            F.relu(x @ self.W.T)          # (B, hidden_dim)
        # Decode: hidden → x (W^T)
        x_hat = h @ self.W + self.b       # (B, n_features)
        return F.relu(x_hat)

def generate_sparse_data(n_samples, n_features, sparsity=0.1):
    """Generate data where each feature is active with probability sparsity."""
    # Features have decreasing importance
    importance = torch.tensor([0.9**i for i in range(n_features)])
    # Sparse activations: Bernoulli mask * uniform value
    mask = (torch.rand(n_samples, n_features) < sparsity).float()
    values = torch.rand(n_samples, n_features)
    return mask * values, importance

# Weighted MSE loss (more important features have higher weight)
def weighted_mse(x, x_hat, importance):
    return ((x - x_hat)**2 * importance).mean()

# Train the toy model
n_features, hidden_dim = 5, 2  # 5 features, 2D hidden space!
torch.manual_seed(42)
toy_model = ToyModelSuperposition(n_features, hidden_dim)
optimizer = torch.optim.Adam(toy_model.parameters(), lr=1e-3)

data_train, importance = generate_sparse_data(50000, n_features)
losses_toy = []

for step in range(2000):
    idx = torch.randint(0, len(data_train), (256,))
    xb = data_train[idx]
    x_hat = toy_model(xb)
    loss = weighted_mse(xb, x_hat, importance)
    optimizer.zero_grad(); loss.backward(); optimizer.step()
    losses_toy.append(loss.item())

print(f"Training complete. Final loss: {losses_toy[-1]:.6f}")

# Visualize the learned feature directions in 2D
W = toy_model.W.detach().numpy()  # (2, 5) — columns are feature directions

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Feature directions in 2D hidden space
ax = axes[0]
theta = np.linspace(0, 2*np.pi, 200)
ax.plot(np.cos(theta), np.sin(theta), 'k--', alpha=0.3, linewidth=1)

colors_f = ['#e74c3c', '#3498db', '#2ecc71', '#9b59b6', '#e67e22']
for i in range(n_features):
    direction = W[:, i] / (np.linalg.norm(W[:, i]) + 1e-8)
    ax.quiver(0, 0, direction[0], direction[1],
             angles='xy', scale_units='xy', scale=1,
             color=colors_f[i], width=0.02, label=f'Feature {i+1}')
    ax.text(direction[0]*1.1, direction[1]*1.1, f'F{i+1}',
            fontsize=12, fontweight='bold', color=colors_f[i], ha='center')

ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5)
ax.set_aspect('equal')
ax.set_title('Learned Feature Directions in 2D\n(5 features packed into 2 dimensions!)', fontweight='bold')
ax.axhline(0, color='k', linewidth=0.5); ax.axvline(0, color='k', linewidth=0.5)

# Check angles between features (should approach ~72° for pentagon = optimal packing)
print("\nAngles between feature directions (ideal pentagon: 72°):")
for i in range(n_features):
    for j in range(i+1, n_features):
        d_i = W[:, i] / np.linalg.norm(W[:, i])
        d_j = W[:, j] / np.linalg.norm(W[:, j])
        angle = np.degrees(np.arccos(np.clip(np.dot(d_i, d_j), -1, 1)))
        print(f"  F{i+1} ∠ F{j+1} = {angle:.1f}°")

# Feature norms (importance → feature should be larger)
norms = np.linalg.norm(W, axis=0)
axes[1].bar(range(1, n_features+1), norms, color=colors_f, alpha=0.85, edgecolor='black')
axes[1].bar(range(1, n_features+1), importance.numpy(),
           color='none', edgecolor='black', linestyle='--', linewidth=2, label='Importance')
axes[1].set_title('Feature Norms vs Importance\n(Important features get more space)', fontweight='bold')
axes[1].set_xlabel('Feature index'); axes[1].set_ylabel('Norm / Importance')
axes[1].legend()

# Training loss
axes[2].semilogy(losses_toy, '#9b59b6', linewidth=2)
axes[2].set_title('Toy Model Training Loss', fontweight='bold')
axes[2].set_xlabel('Step'); axes[2].set_ylabel('Weighted MSE Loss')

plt.tight_layout()
plt.show()

print("\n💡 Key insight: 5 features arranged in a PENTAGON in 2D!")
print("   This is superposition — features interfere slightly but the model")
print("   prefers this over forgetting less-important features entirely.")
print("   Real LLMs have billions of features packed into millions of dimensions.")

In [ ]:
# =============================================================
# 12.4 — Sparse Autoencoder (SAE): The Primary Tool for Mech Interp
# =============================================================

print("Sparse Autoencoder — Disentangling Superposition")
print("Goal: Find the underlying features hidden in superposition")
print()

class SparseAutoencoder(nn.Module):
    """
    SAE for mechanistic interpretability.
    
    Takes model activations → finds sparse decomposition → reconstructs.
    The sparse hidden layer ideally corresponds to interpretable features.
    
    Loss = Reconstruction Loss + λ * Sparsity Loss (L1)
    """
    def __init__(self, d_model, d_hidden, l1_coeff=1e-3):
        super().__init__()
        self.d_model  = d_model
        self.d_hidden = d_hidden
        self.l1_coeff = l1_coeff
        
        # Encoder: d_model → d_hidden
        self.W_enc = nn.Parameter(torch.randn(d_model, d_hidden) * 0.01)
        self.b_enc = nn.Parameter(torch.zeros(d_hidden))
        
        # Decoder: d_hidden → d_model (columns should have unit norm)
        self.W_dec = nn.Parameter(torch.randn(d_hidden, d_model) * 0.01)
        self.b_dec = nn.Parameter(torch.zeros(d_model))
    
    def normalize_decoder(self):
        """Keep decoder columns unit-normed (prevents trivial solutions)."""
        with torch.no_grad():
            self.W_dec.data = F.normalize(self.W_dec.data, dim=1)
    
    def encode(self, x):
        x_centered = x - self.b_dec        # Center around decoder bias
        z = F.relu(x_centered @ self.W_enc + self.b_enc)  # Sparse activations
        return z
    
    def decode(self, z):
        return z @ self.W_dec + self.b_dec
    
    def forward(self, x):
        z = self.encode(x)
        x_hat = self.decode(z)
        
        # Loss
        reconstruction_loss = ((x - x_hat)**2).mean()
        sparsity_loss = z.abs().mean()  # L1 penalty
        total_loss = reconstruction_loss + self.l1_coeff * sparsity_loss
        
        return x_hat, z, total_loss, reconstruction_loss, sparsity_loss

# --- Generate synthetic "model activations" with known ground truth ---
# Simulate: 10 true features, each sparse, compressed into 4D activations
n_true_features = 10
d_model = 4
n_samples = 5000

torch.manual_seed(42)
# Ground truth feature directions (random unit vectors in 4D)
true_directions = F.normalize(torch.randn(n_true_features, d_model), dim=1)

# Generate activations via sparse combinations of true features
sparsity = 0.1
mask = (torch.rand(n_samples, n_true_features) < sparsity).float()
feature_vals = torch.rand(n_samples, n_true_features) * mask
activations = feature_vals @ true_directions  # (n_samples, d_model)
activations = activations + 0.05 * torch.randn_like(activations)  # Add noise

# --- Train SAE ---
sae = SparseAutoencoder(d_model=d_model, d_hidden=20, l1_coeff=5e-3)
opt = torch.optim.Adam(sae.parameters(), lr=1e-3)

sae_losses = {'total': [], 'recon': [], 'sparse': []}
for step in range(3000):
    idx = torch.randint(0, n_samples, (128,))
    x = activations[idx]
    x_hat, z, total, recon, sparse = sae(x)
    opt.zero_grad(); total.backward(); opt.step()
    sae.normalize_decoder()
    sae_losses['total'].append(total.item())
    sae_losses['recon'].append(recon.item())
    sae_losses['sparse'].append(sparse.item())

print(f"SAE training complete!")
print(f"  Reconstruction loss: {sae_losses['recon'][-1]:.5f}")
print(f"  Sparsity loss:       {sae_losses['sparse'][-1]:.5f}")

# Analyze learned features
with torch.no_grad():
    _, z_all, _, _, _ = sae(activations)
    learned_directions = F.normalize(sae.W_dec.data, dim=1)  # (d_hidden, d_model)

# How many SAE features are active? (sparsity)
active_features = (z_all > 0).float().mean(dim=0)  # Fraction of inputs that activate each feature
used_features = (active_features > 0.01).sum().item()

print(f"  SAE hidden dim: 20, Features actively used: {used_features}")
print(f"  Average # active features per sample: {(z_all > 0).float().sum(dim=1).mean().item():.2f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Training curves
ax = axes[0]
ax.semilogy(sae_losses['recon'], '#3498db', linewidth=2, label='Reconstruction')
ax.semilogy(sae_losses['sparse'], '#e74c3c', linewidth=2, label='Sparsity (L1)')
ax.semilogy(sae_losses['total'], '#2ecc71', linewidth=2, label='Total')
ax.set_title('SAE Training Losses', fontweight='bold')
ax.set_xlabel('Step'); ax.legend()

# Feature activation rates
ax2 = axes[1]
active_pct = active_features.numpy() * 100
sorted_idx = np.argsort(active_pct)[::-1]
colors_bar = ['#3498db' if p > 1 else '#bdc3c7' for p in active_pct[sorted_idx]]
ax2.bar(range(len(active_pct)), active_pct[sorted_idx], color=colors_bar)
ax2.axhline(1, color='r', linestyle='--', alpha=0.7, label='1% threshold')
ax2.set_title(f'SAE Feature Activation Rates\n({used_features}/20 features used)', fontweight='bold')
ax2.set_xlabel('Feature index (sorted)'); ax2.set_ylabel('% of inputs that activate')
ax2.legend()

# How well do SAE features recover ground truth?
# Cosine similarity between learned and true directions
sim_matrix = (learned_directions[:used_features] @ true_directions.T).abs().numpy()
im = axes[2].imshow(sim_matrix, cmap='Blues', aspect='auto', vmin=0, vmax=1)
plt.colorbar(im, ax=axes[2])
axes[2].set_title('SAE Feature Recovery\n(|cos sim| between learned & true features)', fontweight='bold')
axes[2].set_xlabel('True features (ground truth)')
axes[2].set_ylabel('SAE learned features')

plt.tight_layout()
plt.show()

print("\n💡 The SAE has learned to approximate the true underlying features!")
print("   In real mech interp: we apply this to actual LLM activations")
print("   Many learned features turn out to be human-interpretable concepts.")

In [ ]:
# =============================================================
# 12.5 — Key Mech Interp Techniques: Probing and Logit Lens
# =============================================================

print("=" * 60)
print("Mechanistic Interpretability Toolkit")
print("=" * 60)

# --- Linear Probing ---
print("\n1. LINEAR PROBING")
print("   Question: Does the residual stream at layer L encode concept C?")
print("   Method: Train a linear classifier on activations to predict C")
print("   If it works → the concept is linearly encoded at that layer!")
print()

# Simulate: Does an intermediate layer encode 'part of speech'?
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

# Create synthetic activations at different layers
n_tokens = 500
# Layer 1: Random (no information)
layer_1_acts = np.random.randn(n_tokens, 64)
# Layer 4: Contains POS information
pos_labels = np.random.randint(0, 3, n_tokens)  # 0=noun, 1=verb, 2=other
signal = np.zeros((n_tokens, 64))
for i, label in enumerate(pos_labels):
    signal[i, :10] = label * 0.5  # POS encoded in first 10 dims
layer_4_acts = np.random.randn(n_tokens, 64) * 0.3 + signal  # Noisy signal

probe_l1 = LogisticRegression(max_iter=500)
probe_l4 = LogisticRegression(max_iter=500)

acc_l1 = cross_val_score(probe_l1, layer_1_acts, pos_labels, cv=5).mean()
acc_l4 = cross_val_score(probe_l4, layer_4_acts, pos_labels, cv=5).mean()

print(f"   Probe accuracy on Layer 1 activations: {acc_l1:.3f} (random baseline ≈ 0.33)")
print(f"   Probe accuracy on Layer 4 activations: {acc_l4:.3f} (much higher → concept is there!)")

# --- Logit Lens ---
print("\n2. LOGIT LENS")
print("   Question: What does the model 'think' the answer is at each layer?")
print("   Method: Project intermediate residual stream → vocabulary at each layer")
print("   Reveals how the model's 'belief' evolves through layers")
print()

# Simulate logit lens with our tiny GPT
tiny_gpt.eval()
# Create a simple token sequence
prompt = "To be or not"
tokens = torch.tensor([encode(prompt)], dtype=torch.long)
T = tokens.shape[1]

# Get activations at each layer
with torch.no_grad():
    # Manually do the forward pass step by step
    positions = torch.arange(T)
    x = tiny_gpt.token_emb(tokens) + tiny_gpt.pos_emb(positions)
    mask = torch.triu(torch.ones(T, T), diagonal=1).bool()
    
    layer_probs = []
    for i, block in enumerate(tiny_gpt.blocks):
        x, _ = block(x, mask=mask)
        # Project to logits at this layer (logit lens!)
        x_norm = tiny_gpt.ln_final(x)
        logits_at_layer = tiny_gpt.lm_head(x_norm)  # (1, T, vocab)
        probs_at_layer = F.softmax(logits_at_layer, dim=-1)
        # What's the top predicted next token at the last position?
        top_token_idx = probs_at_layer[0, -1].argmax().item()
        top_prob = probs_at_layer[0, -1].max().item()
        top_token_char = idx_to_char[top_token_idx]
        layer_probs.append((i+1, top_token_char, top_prob))
    
    print(f"   Prompt: '{prompt}'")
    print(f"   What does the model predict after each layer?")
    for layer, char, prob in layer_probs:
        bar = '█' * int(prob * 30)
        print(f"   Layer {layer}: '{char}' ({prob:.3f}) {bar}")

# --- Summary of Mech Interp Techniques ---
print("\n" + "=" * 60)
print("MECHANISTIC INTERPRETABILITY TOOLKIT SUMMARY")
print("=" * 60)
techniques = [
    ("Probing",             "Train linear classifiers on activations → detect concepts"),
    ("Logit Lens",          "Project intermediate states → vocabulary → track prediction evolution"),
    ("Activation Patching", "Swap activations between runs → find causally important components"),
    ("Attention Patterns",  "Visualize what each head attends to → discover head roles"),
    ("Circuit Analysis",    "Find minimal subgraphs implementing a specific capability"),
    ("Sparse Autoencoders", "Decompose activations into sparse features → interpretable directions"),
    ("Causal Scrubbing",    "Rigorously test circuit hypotheses by ablating paths"),
]
for name, desc in techniques:
    print(f"  {'• ' + name:<25} {desc}")

---
# 🎓 Summary & Your Research Roadmap
---

## What You've Covered

| Section | Core Concepts | Key Takeaway |
|---------|--------------|-------------|
| **Linear Algebra** | Vectors, matrices, SVD, eigenvalues | Neural networks are linear transformations + nonlinearities |
| **Calculus** | Derivatives, chain rule, gradients | Backprop IS the chain rule; gradients tell us how to improve |
| **Probability** | Distributions, MLE, entropy, KL div | Training = maximizing likelihood = minimizing cross-entropy |
| **Optimization** | SGD, momentum, Adam | Adam adapts per-parameter learning rates; almost universally used |
| **Neural Nets** | Perceptron, backprop, activations | Stack linear layers + nonlinearities; learn via gradient descent |
| **Deep Networks** | Residuals, LayerNorm, dropout | Engineering choices that make depth work |
| **RNNs/LSTMs** | Hidden state, gating | Solved sequences before transformers; now largely superseded |
| **Attention** | QKV, scaled dot-product, multi-head | Allows any token to attend to any other — O(T²) but parallelizable |
| **Transformers** | Full architecture, positional encoding | Stacked attention + FFN blocks with residuals |
| **LLMs** | Tokenization, next-token prediction, scaling | Massive scale + simple objective → emergent capabilities |
| **Mech Interp** | Superposition, SAEs, circuits, probing | Research frontier: understanding WHAT models have learned |

---

## 📚 Essential Papers to Read (in order)

**Transformers & LLMs:**
1. **Attention Is All You Need** (Vaswani et al., 2017) — The transformer paper
2. **Language Models are Few-Shot Learners** (Brown et al., 2020) — GPT-3
3. **Training Compute-Optimal LLMs** (Hoffmann et al., 2022) — Chinchilla scaling laws
4. **LLaMA** (Touvron et al., 2023) — Open, efficient LLMs

**Mechanistic Interpretability:**
5. **Zoom In: An Introduction to Circuits** (Olah et al., 2020) — Circuit framework
6. **A Mathematical Framework for Transformer Circuits** (Elhage et al., 2021) — Formal framework
7. **Toy Models of Superposition** (Elhage et al., 2022) — Superposition explained
8. **Towards Monosemanticity: Decomposing LLMs with SAEs** (Bricken et al., 2023) — SAEs
9. **Scaling and Evaluating Sparse Autoencoders** (Gao et al., 2024) — Current state of SAEs
10. **Circuit Discovery with Activation Patching** (Conmy et al., 2023) — Automated circuits

---

## 🗺️ Your Research Roadmap

```
NOW (months 1-2)           → This notebook + Python for Everybody
NEXT (months 3-4)          → Neel Nanda's TransformerLens library
                              Run SAEs on GPT-2 small
                              Reproduce circuits from literature
THEN (months 5-7)          → Deep-dive one specific area:
                              - SAE training & evaluation
                              - Circuit discovery automation
                              - Probing for specific concepts
RESEARCH-READY (month 8+)  → Contribute to open problems:
                              - When does superposition form?
                              - How do features compose?
                              - What do MLP layers store?
                              - Cross-model feature universality
```

---

## 🔧 Tools for Your Research Journey

- **[TransformerLens](https://github.com/neelnanda-io/TransformerLens)** — Neel Nanda's toolkit for mech interp
- **[SAELens](https://github.com/jbloomaus/SAELens)** — Training and analyzing SAEs
- **[CircuitsVis](https://github.com/alan-cooney/CircuitsVis)** — Visualizing circuits
- **[Neuronpedia](https://www.neuronpedia.org)** — Database of SAE features
- **[ARENA curriculum](https://arena3-chapter1-transformer-interp.streamlit.app)** — Structured learning path for mech interp

---

> **A message for your journey:** The math and code in this notebook are tools. The goal is developing *intuition* — an almost physical sense of what these operations mean. Return to sections that feel unclear. Modify the code. Break things. Rebuild them. The researchers in mechanistic interpretability started exactly where you are now.